# Pipeline reproduzível de tese — MBM × MBM+MLP (R1–R8)

**Entrada:** as oito planilhas em `data/`. **Saída:** `resultados_pipeline/` com arquivos `.csv`, pesos `.pt`, estudos Optuna `.db`, figuras bilíngues `.png`, catálogos e sínteses de interpretação.

**Método:** três sementes `(123, 456, 789, 1011, 2027)`, janelas de **12 horas**, divisão temporal **60/20/20**, Protocolos A (um passo observado) e B (recursivo), Optuna restrito a treino/validação, MBB **pareado** em blocos de 12 exemplos para MBM e MBM+MLP (com retreinamento de três sementes por réplica), IC95% percentílicos, teste bootstrap centrado + Holm, DM/HAC para perdas preditivas, ablação, SHAP e figuras 3D individuais de `F_inf` e `k′`.

## Censura — implementação preservada da fonte identificada

O módulo de censura foi extraído do notebook **`pipeline_MBM_MLP_LSTM_combinado_v9_eventos.ipynb`**, disponível nos arquivos do projeto. A classificação pontual e a máquina de estados são preservadas, com critérios: razão `Cin/Cout < 1,20`; ausência de crescimento abrupto `Cin(t) < 1,75 Cin(t−1)`; início forte quando ambos os critérios sugerem interno; início por razão persistentemente interna durante **2 pontos**; persistência até a volta dos dois critérios externos; desconhecidos preservados. `Cout` passa pelo filtro Hampel causal e somente os internos são substituídos por média local.

**Atenção metodológica:** a função de correção da fonte v9 contém fallback que pode consultar observações posteriores **dentro da mesma partição** quando não existem vizinhos anteriores. Como proteção, o pipeline separa os segmentos de censura nos limites de treino, validação e teste. Mesmo assim, essa estratégia não é estritamente causal *dentro de cada partição*. O resultado auditável deve ser citado na tese.

### Antes da execução

1. Instale Python 3.11+ e os pacotes do `requirements.txt`.
2. Execute este notebook na pasta onde está `data/`, sem reorganizar as planilhas.
3. Para fazer primeiro um teste rápido, substitua `CFG = Config()` pela configuração rápida comentada na célula 0.
4. **Não use o conjunto de teste para decidir grupos e hiperparâmetros.** O período de teste dessas bases já foi consultado em análises anteriores: os resultados deste estudo não devem ser chamados de avaliação confirmatória intocada.
5. A execução completa com 48 ajustes finais e milhares de réplicas MBB é computacionalmente exigente. O pipeline grava saídas intermediárias para retomada.

**Cuidado:** reamostrar exemplos pré-formados com MBB não equivale a repetir medição, classificação e re-simulação recursiva em cada réplica; essa limitação é preservada nos relatórios.


## Dependências

Com o ambiente Python ativo, execute no terminal:

```bash
python -m pip install -r requirements.txt
```

Ou descomente a célula de instalação abaixo. As versões efetivamente usadas devem ser registradas com o CSV de auditoria.


In [ ]:
# Execute somente se as bibliotecas ainda não estiverem instaladas:
# %pip install -r requirements.txt

## 0. Configuração e dependências

O bloco abaixo pertence ao protocolo completo.


In [ ]:
from __future__ import annotations
import os, sys, json, csv, hashlib, math, re, time, warnings, platform, random, shutil
from pathlib import Path
from dataclasses import dataclass, asdict
from itertools import combinations
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import least_squares
from scipy import stats
from scipy.stats import norm
import matplotlib
# Para notebooks locais, a figura é exibida com display quando necessário; arquivos com plt.savefig.
import matplotlib.pyplot as plt
from matplotlib import ticker
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from typing import Tuple
import torch
from torch import nn
import optuna
try:
    import shap
    SHAP_AVAILABLE=True
except ImportError:
    shap=None
    SHAP_AVAILABLE=False
from IPython.display import display, Markdown

@dataclass
class Config:
    input_dir: str = 'data'
    output_dir: str = 'resultados_3_sementes_1000_MBB'
    residences: tuple = ('R5','R6','R7')
    fractions: tuple = ('PM1','PM2.5','PM10')
    protocols: tuple = ('A','B')
    seeds: tuple = (123,456,789)
    window: int = 12
    train_prop: float = 0.60
    validation_prop: float = 0.20
    max_trials: int = 20                 # orçamento Optuna por residencia/fração/protocolo
    max_epochs: int = 80
    patience: int = 10
    minibatch: int = 64
    train_refit_epochs: bool = True
    tune_seeds: tuple = (123,456,789)
    mbm_kmax: float = 5.0
    n_mbb: int = 1000
    mbb_block: int = 12
    mbb_protocols: tuple = ('B',)   # A opcional: ('A','B')
    mbb_max_epochs: int = 80
    mbb_adaptive_to: int = 2000     # amplia casos perto do limiar de Holm, se habilitado
    mbb_adaptive: bool = False     # requer refit MLP em CADA nova réplica
    shap_max_samples: int = 45
    shap_background: int = 25
    shap_seed: int = 2026
    plot_dpi: int = 1000
    plot_width_inches: float = 6.4
    run_shap: bool = True
    run_ablation: bool = True
    run_mbb: bool = True
    run_3d: bool = True
    run_figures: bool = True
    update_existing: bool = False
    device: str = 'cpu'          # CPU mais reproduzível; CUDA pode variar
    quick_test: bool = False
    # Não chamar teste já consultado de holdout inteiramente intocado.
    test_previously_inspected: bool = True
    time_policy: str = 'reconstruct_confirmed' # 'validate' ou 'reconstruct_confirmed' (com log)

CFG = Config()
# TESTE RÁPIDO (descomente para validar instalação com baixo custo):
# CFG = Config(residences=('R1',), fractions=('PM1',), protocols=('A','B'),
#              seeds=(123,), tune_seeds=(123,), max_trials=2, max_epochs=5,
#              patience=2, n_mbb=5, mbb_max_epochs=4, plot_dpi=160,
#              run_shap=False, run_ablation=False, quick_test=True)
ROOT=Path.cwd()
DATA=ROOT/CFG.input_dir
OUT=ROOT/CFG.output_dir
OUT.mkdir(parents=True, exist_ok=True)
for sub in ('auditoria','tabelas','modelos','optuna','replicas_mbb','figuras/PT','figuras/EN','insights','dados_processados'):
    (OUT/sub).mkdir(parents=True, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.set_num_threads(1)
print('Raiz:',ROOT,'\nEntrada:',DATA,'\nSaída:',OUT)
print('Execução FULL:',not CFG.quick_test,'| Sementes:',CFG.seeds,'| Bootstrap:',CFG.n_mbb)


## 1. Leitura, auditoria temporal e CENSURA v9 por eventos

O bloco abaixo pertence ao protocolo completo.


In [ ]:
# Dados originais padronizados e algoritmo de censura v9 — código de procedência identificável.
FRACTIONS={'PM1':('cinPM1','coutPM1'),'PM2.5':('cinPM25','coutPM25'),'PM10':('cinPM10','coutPM10')}
BASE_PREDICTORS=['Cout','Tin','Tout','URin','URout','ws','wd_sin','wd_cos','hour_sin','hour_cos']
GROUPS={
 'A1':['Cout'],'A2':['Cout','hour_sin','hour_cos'],
 'B1':['Cout','URout'],'B2':['Cout','ws'],'B3':['Cout','Tin','Tout'],
 'C1':['Cout','URout','ws'],'C2':['Cout','Tin','Tout','URin','URout'],
 'D1':['Cout','Tin','Tout','URin','URout','ws','wd_sin','wd_cos','hour_sin','hour_cos']}
COLORS={'MBM':'#2866A7','Hybrid':'#21825B','observed':'#171717','validation':'#C97432'}
LABEL={
 'PT':{'observed':'Observado (corrigido)','hybrid':'MBM+MLP','t':'Índice temporal',
 'c':'Concentração (µg m⁻³)','rmse':'RMSE (µg m⁻³)', 'f':'Fator de infiltração',
 'k':'Taxa efetiva de remoção (h⁻¹)','x':'Observado (µg m⁻³)','y':'Previsto (µg m⁻³)',
 'train':'Treino','validation':'Validação','test':'Teste','prediction':'Concentração prevista (µg m⁻³)',
 'uncertainty':'IC95% MBB','seed':'DP entre sementes','time':'Índice temporal',
 'difference':'Diferença (MBM − MBM+MLP)'},
 'EN':{'observed':'Observed (corrected)','hybrid':'MBM+MLP','t':'Time index',
 'c':'Concentration (µg m⁻³)','rmse':'RMSE (µg m⁻³)','f':'Infiltration factor',
 'k':'Effective removal rate (h⁻¹)','x':'Observed (µg m⁻³)','y':'Predicted (µg m⁻³)',
 'train':'Training','validation':'Validation','test':'Test','prediction':'Predicted concentration (µg m⁻³)',
 'uncertainty':'95% MBB CI','seed':'Between-seed SD','time':'Time index',
 'difference':'Difference (MBM − MBM+MLP)'}}

def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as file:
  for chunk in iter(lambda:file.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def _time_to_td(x):
 import datetime as _datetime
 if pd.isna(x):return pd.NaT
 if isinstance(x,(int,float,np.integer,np.floating)):
  return pd.to_timedelta(float(x),unit='d')
 if isinstance(x,(_datetime.time,_datetime.datetime,pd.Timestamp)):
  return pd.Timedelta(hours=x.hour,minutes=x.minute,seconds=x.second)
 try:
  p=pd.to_datetime(str(x),errors='raise')
  return pd.Timedelta(hours=p.hour,minutes=p.minute,seconds=p.second)
 except Exception:return pd.NaT

def load_source(res):
 p=DATA/f'{res}_padronizada.xlsx'
 if not p.exists():raise FileNotFoundError(f'Não encontrado: {p}')
 d=pd.read_excel(p,sheet_name=0)
 d.columns=[str(c).strip() for c in d.columns]
 needed=set(['dia','hora']+[x for pair in FRACTIONS.values() for x in pair])
 if needed-set(d.columns):raise ValueError(f'{res}: faltam colunas {needed-set(d.columns)}')
 d=d.dropna(how='all').reset_index(drop=True)
 orig=pd.to_datetime(d['dia'],errors='coerce').dt.normalize()+d['hora'].map(_time_to_td)
 if orig.isna().any():raise ValueError(f'{res}: timestamps inválidos; corrija no arquivo original')
 gaps=orig.diff().dt.total_seconds().div(3600)
 irregular=(gaps.dropna().sub(1).abs()>0.001)
 irregular_n=int(irregular.sum())
 d['timestamp']=orig
 if CFG.time_policy=='reconstruct_confirmed' and irregular_n:
  d['timestamp']=pd.date_range(orig.iloc[0],periods=len(d),freq='h')
  pd.DataFrame({'original':orig,'reconstructed':d.timestamp}).to_csv(OUT/'auditoria'/f'{res}_timestamps.csv',index=False)
 elif CFG.time_policy=='validate' and irregular_n:
  raise ValueError(f'{res}: {irregular_n} intervalos != 1h. Corrija dados ou escolha time_policy="segment".')
 elif CFG.time_policy not in ('validate','segment','reconstruct_confirmed'):
  raise ValueError('time_policy deve ser validate, segment, reconstruct_confirmed')
 d['row_index']=np.arange(len(d))
 # Preservar ordem de medição; lacunas delimitam segmentos, não são preenchidas.
 diff=d.timestamp.diff().dt.total_seconds().div(3600)
 newseg=diff.ne(1.0);newseg.iloc[0]=True
 d['segment_id']=newseg.cumsum().astype(int)-1
 n=len(d);d['split']=np.where(d.row_index<int(n*.6),'train',
                          np.where(d.row_index<int(n*.8),'validation','test'))
 # Segmentação adicional por partição para impedir consulta de pontos futuros no fallback.
 boundaries=d['split'].ne(d['split'].shift(1))
 d['censor_segment_id']=(newseg|boundaries).cumsum().astype(int)
 for col in [*BASE_PREDICTORS,'Tin','Tout','URin','URout','ws','wd','precip']:
  if col not in d:d[col]=np.nan
 for col in [x for pair in FRACTIONS.values() for x in pair]+['Tin','Tout','URin','URout','ws','wd','precip']:
  d[col]=pd.to_numeric(d[col],errors='coerce')
 for col in ['URin','URout']:
  v=d[col].dropna()
  if len(v) and v.quantile(.90)<=1.5:d[col]=100*d[col]
 hod=d.timestamp.dt.hour+d.timestamp.dt.minute/60
 d['hour_sin']=np.sin(2*np.pi*hod/24);d['hour_cos']=np.cos(2*np.pi*hod/24)
 d['wd_sin']=np.sin(np.deg2rad(d.wd));d['wd_cos']=np.cos(np.deg2rad(d.wd))
 d['precip_cat']=np.where(d.precip.notna(),(d.precip>0).astype(float),np.nan)
 d['timestamp_original']=orig
 return d,{'residence':res,'filename':p.name,'sha256':sha256_file(p),'n':len(d),
           'irregular_before_policy':irregular_n,'n_segments':int(d.segment_id.nunique()),
           'n_rain_records':int(d.precip.notna().sum()),'time_policy':CFG.time_policy}

# DEFINIÇÕES LITERAIS DO SCRIPT FONTE "pipeline_MBM_MLP_LSTM_combinado_v9_eventos.ipynb"
RATIO_THRESHOLD=1.20
GROWTH_FACTOR_THRESHOLD=1.75
EVENT_SUSTAINED_RATIO_POINTS=2
EVENT_SENSITIVITY_POINTS=(2,3,4)
LOCAL_HALF_WINDOW=3
OUTLIER_Z=3.5
PM_FRACTIONS={'MP1':FRACTIONS['PM1'],'MP2.5':FRACTIONS['PM2.5'],'MP10':FRACTIONS['PM10']}

# --- INÍCIO: BLOCO DE CENSURA V9, SEM ALTERAÇÃO DE FUNÇÕES ---
def causal_hampel_replace(values: np.ndarray, segments: np.ndarray, half_window: int = 3, z: float = 3.5):
    values = np.asarray(values, float)
    segments = np.asarray(segments)
    corrected = values.copy()
    flags = np.zeros(len(values), dtype=bool)
    history_length = max(8 * half_window, 24)
    minimum_history = max(2 * half_window, 8)

    for i in range(len(values)):
        same_segment_history = [
            j for j in range(max(0, i - history_length), i)
            if segments[j] == segments[i] and np.isfinite(values[j])
        ]
        if len(same_segment_history) < minimum_history or not np.isfinite(values[i]):
            continue
        history = values[same_segment_history]
        median = np.median(history)
        mad = np.median(np.abs(history - median))
        scale = 1.4826 * mad
        if scale > 0 and abs(values[i] - median) > z * scale:
            flags[i] = True
            corrected[i] = float(np.mean(history[-min(len(history), 2 * half_window):]))
    return corrected, flags


def pointwise_source_evidence(cin: np.ndarray, cout: np.ndarray, segments: np.ndarray,
                              ratio_threshold: float = RATIO_THRESHOLD,
                              growth_factor: float = GROWTH_FACTOR_THRESHOLD) -> pd.DataFrame:
    """Evidência pontual auditável; não altera medições nem cria eventos."""
    cin = np.asarray(cin, float)
    cout = np.asarray(cout, float)
    segments = np.asarray(segments)
    rows = []

    for i in range(len(cin)):
        row = dict(
            pointwise_class="desconhecido",
            pointwise_reason="",
            ratio_cin_cout=np.nan,
            growth_factor_observed=np.nan,
            ratio_external=False,
            no_sharp_growth=False,
            context_valid=False,
        )
        if not (np.isfinite(cin[i]) and np.isfinite(cout[i])):
            row["pointwise_reason"] = "dados_nao_finitos"
            rows.append(row)
            continue

        row["ratio_cin_cout"] = cin[i] / max(cout[i], 1e-12)
        if i == 0 or segments[i] != segments[i - 1] or not np.isfinite(cin[i - 1]):
            row["pointwise_reason"] = "sem_contexto_temporal"
            rows.append(row)
            continue

        row["context_valid"] = True
        row["growth_factor_observed"] = cin[i] / max(cin[i - 1], 1e-12)
        ratio_external = bool(row["ratio_cin_cout"] < ratio_threshold)
        no_sharp_growth = bool(cin[i] < growth_factor * cin[i - 1])
        row["ratio_external"] = ratio_external
        row["no_sharp_growth"] = no_sharp_growth

        if ratio_external and no_sharp_growth:
            row["pointwise_class"] = "externo"
            row["pointwise_reason"] = "dois_criterios_externos"
        elif (not ratio_external) and (not no_sharp_growth):
            row["pointwise_class"] = "interno"
            row["pointwise_reason"] = "dois_criterios_internos"
        elif ratio_external:
            row["pointwise_reason"] = "razao_externa_crescimento_abrupto"
        else:
            row["pointwise_reason"] = "razao_interna_sem_crescimento_abrupto"
        rows.append(row)

    return pd.DataFrame(rows)


def classify_sources_by_events(cin: np.ndarray, cout: np.ndarray, segments: np.ndarray,
                               sustained_ratio_points: int = EVENT_SUSTAINED_RATIO_POINTS,
                               ratio_threshold: float = RATIO_THRESHOLD,
                               growth_factor: float = GROWTH_FACTOR_THRESHOLD) -> pd.DataFrame:
    """Máquina de estados causal: início, persistência e encerramento."""
    if int(sustained_ratio_points) < 2:
        raise ValueError("sustained_ratio_points deve ser >= 2")

    segments = np.asarray(segments)
    evidence = pointwise_source_evidence(cin, cout, segments, ratio_threshold, growth_factor)
    pointwise = evidence["pointwise_class"].to_numpy(object)
    reasons = evidence["pointwise_reason"].to_numpy(object)
    final_labels = pointwise.copy()
    event_ids = np.full(len(pointwise), -1, dtype=int)
    phases = np.full(len(pointwise), "", dtype=object)
    onset_trigger = np.full(len(pointwise), "", dtype=object)
    end_reason = np.full(len(pointwise), "", dtype=object)

    active_event = -1
    next_event_id = 0
    high_ratio_streak = 0

    for i in range(len(pointwise)):
        new_segment = i == 0 or segments[i] != segments[i - 1]
        if new_segment:
            active_event = -1
            high_ratio_streak = 0

        context_valid = bool(evidence.at[i, "context_valid"])
        ratio_internal = context_valid and not bool(evidence.at[i, "ratio_external"])
        high_ratio_streak = high_ratio_streak + 1 if ratio_internal else 0

        if active_event >= 0:
            if pointwise[i] == "externo":
                active_event = -1
                final_labels[i] = "externo"
            elif reasons[i] in {"dados_nao_finitos", "sem_contexto_temporal"}:
                active_event = -1
                final_labels[i] = "desconhecido"
            else:
                final_labels[i] = "interno"
                event_ids[i] = active_event
            continue

        strong_onset = pointwise[i] == "interno"
        persistent_onset = (
            reasons[i] == "razao_interna_sem_crescimento_abrupto"
            and high_ratio_streak >= int(sustained_ratio_points)
        )
        if strong_onset or persistent_onset:
            active_event = next_event_id
            next_event_id += 1
            final_labels[i] = "interno"
            event_ids[i] = active_event
            onset_trigger[i] = "dois_criterios_internos" if strong_onset else "razao_interna_persistente"

    for event_id in range(next_event_id):
        idx = np.where(event_ids == event_id)[0]
        if not len(idx):
            raise AssertionError("Evento criado sem pontos")
        start, end = int(idx[0]), int(idx[-1])
        if len(idx) == 1:
            phases[start] = "inicio_encerramento"
        else:
            phases[idx] = "persistencia"
            phases[start] = "inicio"
            phases[end] = "encerramento"
        if end + 1 < len(pointwise) and segments[end + 1] == segments[end] and pointwise[end + 1] == "externo":
            end_reason[end] = "retorno_externo"
        elif end + 1 >= len(pointwise) or segments[end + 1] != segments[end]:
            end_reason[end] = "fim_segmento"
        else:
            end_reason[end] = "interrupcao_contexto"

    result = evidence.copy()
    result["source_class"] = final_labels
    result["event_id"] = event_ids
    result["event_phase"] = phases
    result["event_onset_trigger"] = onset_trigger
    result["event_end_reason"] = end_reason
    result["reclassified_by_event"] = pointwise != final_labels

    # Invariantes: eventos são internos, não atravessam segmentos e preservam
    # todos os pontos que continuam desconhecidos.
    if not np.all(result.loc[result["event_id"] >= 0, "source_class"] == "interno"):
        raise AssertionError("Há evento com classe final não interna")
    for event_id in result.loc[result["event_id"] >= 0, "event_id"].unique():
        idx = np.where(result["event_id"].to_numpy() == event_id)[0]
        if len(np.unique(segments[idx])) != 1 or not np.all(np.diff(idx) == 1):
            raise AssertionError("Evento atravessou lacuna ou deixou de ser contíguo")
        if np.sum(np.isin(result.loc[idx, "event_phase"], ["inicio", "inicio_encerramento"])) != 1:
            raise AssertionError("Evento sem início único")
        if np.sum(np.isin(result.loc[idx, "event_phase"], ["encerramento", "inicio_encerramento"])) != 1:
            raise AssertionError("Evento sem encerramento único")
    return result


def replace_internal_by_local_mean(cin: np.ndarray, labels: np.ndarray, segments: np.ndarray,
                                   half_window: int = LOCAL_HALF_WINDOW) -> Tuple[np.ndarray, np.ndarray]:
    cin = np.asarray(cin, float)
    labels = np.asarray(labels)
    segments = np.asarray(segments)
    corrected = cin.copy()
    internal = labels == "interno"
    replaced = np.zeros(len(cin), dtype=bool)

    for i in np.where(internal)[0]:
        previous = [
            j for j in range(max(0, i - 2 * half_window), i)
            if segments[j] == segments[i] and labels[j] != "interno" and np.isfinite(cin[j])
        ]
        candidates = previous[-2 * half_window:]
        if not candidates:
            following = [
                j for j in range(i + 1, min(len(cin), i + 2 * half_window + 1))
                if segments[j] == segments[i] and labels[j] != "interno" and np.isfinite(cin[j])
            ]
            candidates = following[:2 * half_window]
        if not candidates:
            neighbors = [
                j for j in range(len(cin))
                if j != i and segments[j] == segments[i] and np.isfinite(cin[j])
            ]
            candidates = sorted(neighbors, key=lambda j: abs(j - i))[:2 * half_window]
        if candidates:
            corrected[i] = float(np.mean(cin[candidates]))
            replaced[i] = True
    return corrected, replaced


def prepare_case_dataframe(df_house: pd.DataFrame, pm: str,
                           sustained_ratio_points: int = EVENT_SUSTAINED_RATIO_POINTS) -> pd.DataFrame:
    cin_col, cout_col = PM_FRACTIONS[pm]
    out = df_house.copy()
    segments = out["segment_id"].to_numpy()
    cin_raw = out[cin_col].to_numpy(float)
    cout_clean, cout_outlier = causal_hampel_replace(
        out[cout_col].to_numpy(float), segments,
        half_window=LOCAL_HALF_WINDOW, z=OUTLIER_Z,
    )
    classification = classify_sources_by_events(
        cin_raw, cout_clean, segments, sustained_ratio_points=sustained_ratio_points
    )
    labels = classification["source_class"].to_numpy(object)
    cin_calibration, internal_replaced = replace_internal_by_local_mean(
        cin_raw, labels, segments, half_window=LOCAL_HALF_WINDOW
    )

    internal_mask = labels == "interno"
    unknown_mask = labels == "desconhecido"
    if not internal_replaced[internal_mask].all():
        raise AssertionError("Há ponto interno de evento sem substituição por média local")
    if not np.allclose(cin_calibration[unknown_mask], cin_raw[unknown_mask], equal_nan=True):
        raise AssertionError("Um ponto desconhecido foi alterado indevidamente")
    if not np.all(labels[classification["pointwise_class"].to_numpy(object) == "interno"] == "interno"):
        raise AssertionError("Um ponto interno confirmado foi perdido pela máquina de eventos")

    out["cin_raw"] = cin_raw
    out["cout_raw"] = out[cout_col].to_numpy(float)
    out["cout_clean"] = cout_clean
    out["cout_outlier"] = cout_outlier
    for column in classification.columns:
        out[column] = classification[column].to_numpy()
    out["cin_calibration"] = cin_calibration
    out["internal_replaced"] = internal_replaced
    return out


def summarize_internal_events(prepared: pd.DataFrame, house: str, pm: str) -> pd.DataFrame:
    rows = []
    event_ids = sorted(int(x) for x in prepared.loc[prepared["event_id"] >= 0, "event_id"].unique())
    for event_id in event_ids:
        event = prepared.loc[prepared["event_id"] == event_id]
        start = int(event.index.min())
        end = int(event.index.max())
        rows.append(dict(
            casa=house,
            pm=pm,
            event_id=event_id,
            segment_id=int(event["segment_id"].iloc[0]),
            inicio=event["timestamp"].iloc[0],
            encerramento=event["timestamp"].iloc[-1],
            duracao_h=len(event),
            gatilho=event["event_onset_trigger"].iloc[0],
            motivo_encerramento=event["event_end_reason"].iloc[-1],
            censurado_a_direita=event["event_end_reason"].iloc[-1] != "retorno_externo",
            internos_pontuais=int((event["pointwise_class"] == "interno").sum()),
            desconhecidos_reclassificados=int(
                ((event["pointwise_class"] == "desconhecido") & event["reclassified_by_event"]).sum()
            ),
            media_cin_raw=float(event["cin_raw"].mean()),
            media_cin_calibration=float(event["cin_calibration"].mean()),
        ))
    return pd.DataFrame(rows)



# --- FIM DO BLOCO V9 ---

def prepared_case(df,res,fraction):
 original=df.copy()
 # O algoritmo é aplicado por partições para evitar fallback com dados do futuro em outra partição.
 original['segment_id']=original['censor_segment_id']
 raw=prepare_case_dataframe(original,{'PM1':'MP1','PM2.5':'MP2.5','PM10':'MP10'}[fraction])
 raw['Cin_orig']=raw['cin_raw'];raw['Cout']=raw['cout_clean']
 raw['Cin_corrected']=raw['cin_calibration'];raw['source_event']=raw['source_class']
 raw['Cin_prev_corrected']=raw['Cin_corrected'].shift(1)
 # Produzir indicadores completos e verificar que os desconhecidos mantêm o original.
 unk=raw.source_class.eq('desconhecido').to_numpy()
 assert np.allclose(raw.loc[unk,'Cin_orig'],raw.loc[unk,'Cin_corrected'],equal_nan=True)
 out=OUT/'dados_processados'/f'{res}_{fraction.replace(".","")}.csv'
 raw.to_csv(out,index=False)
 return raw

SOURCE_AUDIT=[];DATASETS={}
for residence in CFG.residences:
 frame,audit=load_source(residence)
 DATASETS[residence]=frame;SOURCE_AUDIT.append(audit)
pd.DataFrame(SOURCE_AUDIT).to_csv(OUT/'auditoria'/'fontes_e_cronologia.csv',index=False)
print(pd.DataFrame(SOURCE_AUDIT).to_string(index=False))


## 2. Balanço de massa e MBM+MLP

O bloco abaixo pertence ao protocolo completo.


In [ ]:
def set_seed(seed):
 random.seed(int(seed));np.random.seed(int(seed));torch.manual_seed(int(seed))
 if torch.cuda.is_available():torch.cuda.manual_seed_all(int(seed))

def make_examples(frame,features):
 X=frame[features].to_numpy(dtype=float)
 cin=frame['Cin_corrected'].to_numpy(dtype=float)
 cout=frame['Cout'].to_numpy(dtype=float)
 tlist=[];window=[];prev=[];ext=[];Y=[];spl=[]
 for t in range(CFG.window-1,len(frame)):
  if frame['segment_id'].iloc[t]!=frame['segment_id'].iloc[t-CFG.window+1]:continue
  if t<1 or not all(np.isfinite([cin[t-1],cin[t],cout[t]])):continue
  # Falta de meteo: imputar com parâmetros do treino; não imputar concentrações.
  tlist.append(t)
  window.append(X[t-CFG.window+1:t+1].flatten())
  prev.append(cin[t-1]);ext.append(cout[t]);Y.append(cin[t]);spl.append(frame.split.iloc[t])
 if not window:raise ValueError('Nenhuma janela temporal elegível.')
 return {'X':np.asarray(window,dtype=np.float32),'t':np.asarray(tlist,dtype=int),
         'prev':np.asarray(prev,dtype=np.float32),'cout':np.asarray(ext,dtype=np.float32),
         'y':np.asarray(Y,dtype=np.float32),'split':np.asarray(spl),'features':list(features)}

def subset(ex,where):
 mask=np.isin(ex['split'], list(where))
 return np.where(mask)[0]

def fit_scalers(X,train_idx):
 # Ajustar somente ao treino; em bootstraps, somente à amostra reamostrada.
 im=SimpleImputer(strategy='median',keep_empty_features=True)
 sc=StandardScaler()
 Xt=im.fit_transform(X[train_idx]);sc.fit(Xt)
 return im,sc

def transform_ex(ex,im,sc):
 return sc.transform(im.transform(ex['X'])).astype(np.float32)

def mbm_eq(cprev,cout,F,k):
 a=np.exp(-np.asarray(k))
 return a*np.asarray(cprev)+(1-a)*np.asarray(F)*np.asarray(cout)

def fit_mbm(ex,idx):
 p=ex['prev'][idx];o=ex['cout'][idx];y=ex['y'][idx]
 def residual(z):return mbm_eq(p,o,z[0],z[1])-y
 best=None
 for f0,k0 in ((.3,.3),(.7,.7),(.9,1.5)):
  result=least_squares(residual,[f0,k0],bounds=([0.,0.],[1.,CFG.mbm_kmax]),max_nfev=250)
  if best is None or np.linalg.norm(result.fun)<best[0]:best=(np.linalg.norm(result.fun),result.x)
 return {'Finf':float(best[1][0]),'kprime':float(best[1][1])}

class PhysicalMLP(nn.Module):
 def __init__(self,n_in,hidden=32,dropout=.10):
  super().__init__()
  self.layers=nn.Sequential(nn.Linear(n_in,hidden),nn.ReLU(),nn.Dropout(dropout),
                            nn.Linear(hidden,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,2))
 def forward(self,x,prev=None,cout=None):
  z=self.layers(x)
  f=torch.sigmoid(z[:,0]);k=torch.nn.functional.softplus(z[:,1])
  if prev is None or cout is None:return f,k
  a=torch.exp(-k)
  pred=a*prev+(1-a)*f*cout
  return pred,f,k

def recursive_prediction(pred_params,ex,indices):
 """Usa concentração observada/corrigida apenas para inicializar o início do bloco.
 Exógenas Cout e meteorologia são assumidas conhecidas; não é forecast desses sinais.
 """
 indices=np.asarray(indices,dtype=int)
 if not len(indices):return np.empty(0)
 f,k=pred_params
 predictions=np.empty(len(indices),dtype=float)
 last=None;prior_t=None
 for j,i in enumerate(indices):
  t=int(ex['t'][i]);p=float(ex['prev'][i]) if last is None or prior_t!=t-1 else last
  last=float(mbm_eq(p,float(ex['cout'][i]),float(f[i]),float(k[i])))
  predictions[j]=last;prior_t=t
 return predictions

def model_parameters(model,X,device='cpu',batch=2048):
 model.eval();f=[];k=[]
 with torch.no_grad():
  for start in range(0,len(X),batch):
   ff,kk=model(torch.tensor(X[start:start+batch],device=device))
   f.append(ff.cpu().numpy());k.append(kk.cpu().numpy())
 return np.concatenate(f),np.concatenate(k)

def mean_metrics(y,p):
 y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
 ok=np.isfinite(y)&np.isfinite(p);y=y[ok];p=p[ok]
 if not len(y):return {'n':0,'RMSE':np.nan,'MAE':np.nan,'R2':np.nan,'NRMSE':np.nan,'bias':np.nan}
 rmse=float(np.sqrt(np.mean((y-p)**2)))
 return {'n':len(y),'RMSE':rmse,'MAE':float(np.mean(np.abs(y-p))),
         'R2':float(r2_score(y,p)) if len(y)>1 and np.var(y)>0 else np.nan,
         'NRMSE':rmse/max(float(np.ptp(y)),1e-8),'bias':float(np.mean(p-y))}

def evaluate_full_params(ex,params,idx,protocol):
 f,k=params
 if protocol=='A':pred=mbm_eq(ex['prev'][idx],ex['cout'][idx],f[idx],k[idx])
 else:pred=recursive_prediction((f,k),ex,idx)
 return np.asarray(pred,dtype=float),mean_metrics(ex['y'][idx],pred)

def train_net(ex,train_idx,val_idx,params,seed,epochs=None,early_stop=True,store_history=True):
 set_seed(seed)
 im,sc=fit_scalers(ex['X'],train_idx)
 X=transform_ex(ex,im,sc)
 dev=CFG.device if CFG.device=='cuda' and torch.cuda.is_available() else 'cpu'
 net=PhysicalMLP(X.shape[1],int(params['hidden']),float(params['dropout'])).to(dev)
 optimizer=torch.optim.Adam(net.parameters(),lr=float(params['lr']),weight_decay=float(params.get('weight_decay',1e-5)))
 tX=torch.as_tensor(X[train_idx],dtype=torch.float32,device=dev)
 tp=torch.as_tensor(ex['prev'][train_idx],dtype=torch.float32,device=dev)
 tc=torch.as_tensor(ex['cout'][train_idx],dtype=torch.float32,device=dev)
 ty=torch.as_tensor(ex['y'][train_idx],dtype=torch.float32,device=dev)
 history=[];best_loss=float('inf');best_epoch=1;best_state=None;stale=0
 n_epochs=int(epochs if epochs is not None else CFG.max_epochs)
 for epoch in range(1,n_epochs+1):
  net.train()
  perm=torch.randperm(len(train_idx),device=dev)
  for batch_idx in perm.split(CFG.minibatch):
   optimizer.zero_grad(set_to_none=True)
   pred,_,_=net(tX[batch_idx],tp[batch_idx],tc[batch_idx])
   loss=torch.mean((pred-ty[batch_idx])**2)
   loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),5.)
   optimizer.step()
  net.eval()
  with torch.no_grad():
   predtrain,_,_=net(tX,tp,tc)
   trainrmse=float(torch.sqrt(torch.mean((predtrain-ty)**2)).item())
  valrmse=float('nan')
  if len(val_idx):
   f,k=model_parameters(net,X,dev)
   valpred,_=evaluate_full_params(ex,(f,k),val_idx,params.get('protocol','B'))
   valrmse=mean_metrics(ex['y'][val_idx],valpred)['RMSE']
  if store_history:history.append({'epoch':epoch,'train_RMSE_A':trainrmse,'validation_RMSE':valrmse})
  if not np.isnan(valrmse) and valrmse<best_loss-1e-7:
   best_loss=valrmse;best_epoch=epoch
   best_state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()};stale=0
  else:stale+=1
  if early_stop and len(val_idx) and stale>=CFG.patience:break
 if best_state is not None and early_stop:net.load_state_dict(best_state)
 return {'net':net,'imputer':im,'scaler':sc,'history':history,
         'best_epoch':best_epoch,'best_validation':best_loss,'X':X}

def choose_group_names(res):
 names=list(GROUPS.keys())
 return names

def candidate_features(group,with_precip,res):
 feat=list(GROUPS[group])
 if with_precip and res in ('R5','R6','R7','R8'):
  feat+=['precip_cat']
 return feat


## 3. Otimização Optuna, três sementes, avaliação temporal

O bloco abaixo pertence ao protocolo completo.


In [ ]:
def tune_case(frame,res,fraction,protocol):
 """Optuna apenas treino 60% e validação 20%. Nunca usa teste para escolhas."""
 studyfile=OUT/'optuna'/f'{res}_{fraction.replace(".","")}_{protocol}.db'
 study=optuna.create_study(study_name='mbm_mlp',storage='sqlite:///'+studyfile.as_posix(),
         load_if_exists=True,direction='minimize',
         sampler=optuna.samplers.TPESampler(seed=2026),
         pruner=optuna.pruners.MedianPruner(n_startup_trials=8))
 def objective(trial):
  group=trial.suggest_categorical('group',choose_group_names(res))
  withrain=trial.suggest_categorical('precip_cat',[False,True]) if res in ('R5','R6','R7','R8') else False
  feat=candidate_features(group,withrain,res)
  ex=make_examples(frame,feat)
  tr=subset(ex,('train',));va=subset(ex,('validation',))
  hidden=trial.suggest_categorical('hidden',[16,32,64])
  dropout=trial.suggest_float('dropout',0,.35,step=.05)
  lr=trial.suggest_float('lr',3e-4,3e-3,log=True)
  wd=trial.suggest_float('weight_decay',1e-7,1e-3,log=True)
  para={'hidden':hidden,'dropout':dropout,'lr':lr,'weight_decay':wd,'protocol':protocol}
  metric=[];epochs=[]
  for seed in CFG.tune_seeds:
   m=train_net(ex,tr,va,para,seed,epochs=CFG.max_epochs,early_stop=True,store_history=False)
   metric.append(m['best_validation']);epochs.append(m['best_epoch'])
  trial.set_user_attr('epochs_selected',int(np.median(epochs)))
  trial.set_user_attr('sd_seeds',float(np.std(metric,ddof=1)) if len(metric)>1 else 0.)
  return float(np.mean(metric))
 missing=max(0,CFG.max_trials-len([t for t in study.trials if t.state.is_finished()]))
 if missing:study.optimize(objective,n_trials=missing,n_jobs=1)
 best=study.best_trial
 config=dict(best.params)
 config['protocol']=protocol
 config['epochs']=int(best.user_attrs['epochs_selected'])
 config['features']=candidate_features(config['group'],config.get('precip_cat',False),res)
 config['best_val_RMSE']=float(best.value)
 config['n_trials']=len(study.trials)
 # Curvas de treinamento e validação reais, sem confundir com RMSE por tentativa.
 ex=make_examples(frame,config['features'])
 tr=subset(ex,('train',));va=subset(ex,('validation',))
 curves=[]
 for seed in CFG.seeds:
  fitted=train_net(ex,tr,va,config,seed,epochs=CFG.max_epochs,early_stop=True,store_history=True)
  for row in fitted['history']:
   curves.append({'seed':seed,**row})
 pd.DataFrame(curves).to_csv(OUT/'optuna'/f'{res}_{fraction.replace(".","")}_{protocol}_learning_curves.csv',index=False)
 with open(OUT/'optuna'/f'{res}_{fraction.replace(".","")}_{protocol}_selected.json','w',encoding='utf8') as f:
  json.dump(config,f,indent=2,ensure_ascii=False)
 study.trials_dataframe().to_csv(OUT/'optuna'/f'{res}_{fraction.replace(".","")}_{protocol}_trials.csv',index=False)
 return config,ex


def fit_final_case(frame,res,fraction,protocol,config,ex):
 tr=subset(ex,('train',));va=subset(ex,('validation',));de=subset(ex,('train','validation'))
 te=subset(ex,('test',))
 mbm=fit_mbm(ex,de)
 fmbm=np.full(len(ex['t']),mbm['Finf']);kmbm=np.full(len(ex['t']),mbm['kprime'])
 mbm_preds={p:evaluate_full_params(ex,(fmbm,kmbm),ix,protocol)[0] for p,ix in
            [('train',tr),('validation',va),('test',te)]}
 prediction_rows=[];metric_rows=[];param_rows=[]
 for seed in CFG.seeds:
  fname=OUT/'modelos'/f'{res}_{fraction.replace(".","")}_{protocol}_seed{seed}.pt'
  if fname.exists() and not CFG.update_existing:
   state=torch.load(fname,map_location='cpu',weights_only=False)
   im=state['imputer'];sc=state['scaler'];model=PhysicalMLP(ex['X'].shape[1],
      int(config['hidden']),float(config['dropout']))
   model.load_state_dict(state['model_state'])
   X=transform_ex(ex,im,sc)
  else:
   fit=train_net(ex,de,np.empty(0,dtype=int),config,seed,epochs=config['epochs'],early_stop=False)
   model=fit['net'];im=fit['imputer'];sc=fit['scaler'];X=fit['X']
   torch.save({'model_state':model.state_dict(),'imputer':im,'scaler':sc,
    'config':config,'source_sha256':sha256_file(DATA/f'{res}_padronizada.xlsx'),
    'seed':seed,'torch_version':torch.__version__},fname)
  f,k=model_parameters(model,X)
  for split,idx in [('train',tr),('validation',va),('test',te)]:
   p_mlp,m_mlp=evaluate_full_params(ex,(f,k),idx,protocol)
   p_mbm=mbm_preds[split]
   m_mbm=mean_metrics(ex['y'][idx],p_mbm)
   for modelname,metrics in [('MBM',m_mbm),('MBM_MLP',m_mlp)]:
    metric_rows.append({'residence':res,'fraction':fraction,'protocol':protocol,
     'seed':seed,'period':split,'model':modelname,'status':
     'resubstitution_after_80pct_refit' if split!='test' else 'previously_inspected_temporal_test',**metrics})
   for j,i in enumerate(idx):
    prediction_rows.append({'residence':res,'fraction':fraction,'protocol':protocol,
      'seed':seed,'period':split,'row_index':int(ex['t'][i]),'C_observed_corrected':float(ex['y'][i]),
      'Cout':float(ex['cout'][i]),'Cprev_corrected':float(ex['prev'][i]),
      'MBM':float(p_mbm[j]),'MBM_MLP':float(p_mlp[j]),
      'Finf_MBM':mbm['Finf'],'kprime_MBM':mbm['kprime'],
      'Finf_MLP':float(f[i]),'kprime_MLP':float(k[i])})
   if split=='test':
    param_rows.append({'residence':res,'fraction':fraction,'protocol':protocol,'seed':seed,
       'Finf_MLP_temporal_mean':float(np.mean(f[idx])),'Finf_MLP_temporal_SD':float(np.std(f[idx],ddof=1)) if len(idx)>1 else np.nan,
       'kprime_MLP_temporal_mean':float(np.mean(k[idx])),'kprime_MLP_temporal_SD':float(np.std(k[idx],ddof=1)) if len(idx)>1 else np.nan,
       'Finf_MBM':mbm['Finf'],'kprime_MBM':mbm['kprime']})
 stem=f'{res}_{fraction.replace(".","")}_{protocol}'
 pd.DataFrame(prediction_rows).to_csv(OUT/'tabelas'/f'{stem}_predicoes.csv',index=False)
 pd.DataFrame(metric_rows).to_csv(OUT/'tabelas'/f'{stem}_metricas.csv',index=False)
 pd.DataFrame(param_rows).to_csv(OUT/'tabelas'/f'{stem}_parametros.csv',index=False)
 return pd.DataFrame(prediction_rows),pd.DataFrame(metric_rows),pd.DataFrame(param_rows)

all_metrics=[];all_prediction=[];all_params=[];all_configs=[]
case_inventory=[]
for res in CFG.residences:
 data=DATASETS[res]
 for fraction in CFG.fractions:
  frame=prepared_case(data,res,fraction)
  case_inventory.append({'residence':res,'fraction':fraction,'n':len(frame),
   'n_internal_event':int((frame.source_event=='interno').sum()),
   'n_unknown':int((frame.source_event=='desconhecido').sum()),
   'n_corrected':int((frame.Cin_orig.ne(frame.Cin_corrected)&frame.Cin_orig.notna()).sum())})
  for protocol in CFG.protocols:
   stem=f'{res}_{fraction.replace(".","")}_{protocol}'
   csvpred=OUT/'tabelas'/f'{stem}_predicoes.csv'
   csvmet=OUT/'tabelas'/f'{stem}_metricas.csv'
   csvpar=OUT/'tabelas'/f'{stem}_parametros.csv'
   js=OUT/'optuna'/f'{stem}_selected.json'
   if all(p.exists() for p in [csvpred,csvmet,csvpar,js]) and not CFG.update_existing:
    pred=pd.read_csv(csvpred);met=pd.read_csv(csvmet);par=pd.read_csv(csvpar)
    selected=json.loads(js.read_text(encoding='utf8'))
   else:
    print('Optuna + treino final:',res,fraction,protocol,flush=True)
    selected,ex=tune_case(frame,res,fraction,protocol)
    pred,met,par=fit_final_case(frame,res,fraction,protocol,selected,ex)
   all_prediction.append(pred);all_metrics.append(met);all_params.append(par)
   all_configs.append({'residence':res,'fraction':fraction,'protocol':protocol,**selected})
   print('Concluído:',stem,flush=True)
PRED=pd.concat(all_prediction,ignore_index=True)
MET=pd.concat(all_metrics,ignore_index=True)
PAR=pd.concat(all_params,ignore_index=True)
CONF=pd.DataFrame(all_configs)
INV=pd.DataFrame(case_inventory)
for name,data in [('previsoes_todas.csv',PRED),('metricas_todas.csv',MET),
                  ('parametros_todos.csv',PAR),('configuracoes_optuna.csv',CONF),
                  ('inventario_censura.csv',INV)]:
 data.to_csv(OUT/'tabelas'/name,index=False)
print('Concluído:',PRED.shape,MET.shape,PAR.shape)


## 4. MBB pareado (dois modelos) e testes estatísticos

O bloco abaixo pertence ao protocolo completo.


In [ ]:
def mbb_indices(n,block,rng):
 """Circular moving-block bootstrap of chronological example indices, same draws for both models."""
 if n<2:raise ValueError('Série muito curta para MBB')
 b=min(block,n);starts=rng.integers(0,n,size=int(np.ceil(n/b)))
 return np.concatenate([np.mod(s+np.arange(b),n) for s in starts])[:n]

def sample_summary(values):
 z=np.asarray(values,dtype=float);z=z[np.isfinite(z)]
 if not len(z):return {'mean':np.nan,'SD':np.nan,'CI95_lower':np.nan,'CI95_upper':np.nan,'B':0}
 return {'mean':float(z.mean()),'SD':float(z.std(ddof=1)) if len(z)>1 else np.nan,
         'CI95_lower':float(np.percentile(z,2.5)),'CI95_upper':float(np.percentile(z,97.5)),'B':len(z)}

def centered_bootstrap_test(boot_diffs,observed):
 """p de teste bilateral centrado na estimativa original (não simples contagem de sinais)."""
 b=np.asarray(boot_diffs,dtype=float)
 hit=int(np.sum(np.abs(b-observed)>=abs(observed)))
 B=len(b)
 p=(hit+1)/(B+1)
 # Probabilidade de excedência da simulação de bootstrap (IC exato binomial)
 lo=stats.beta.ppf(.025,hit,B-hit+1) if hit>0 else 0.
 hi=stats.beta.ppf(.975,hit+1,B-hit) if hit<B else 1.
 eps=np.percentile(np.abs(b-observed),95)
 return {'p_bootstrap':float(p),'exceedances':hit,'B':B,
         'MC_p_lower95':float(lo),'MC_p_upper95':float(hi),
         'delta_obs':float(observed),'basic_CI95_lower':float(2*observed-np.percentile(b,97.5)),
         'basic_CI95_upper':float(2*observed-np.percentile(b,2.5)),
         'symmetric_CI95_lower':float(observed-eps),'symmetric_CI95_upper':float(observed+eps)}

def holm_adjust(pvals):
 p=np.asarray(pvals,dtype=float)
 if not len(p):return np.empty(0)
 rank=np.argsort(p);adjusted=np.empty(len(p))
 adj_sorted=np.maximum.accumulate((len(p)-np.arange(len(p)))*p[rank])
 adjusted[rank]=np.minimum(1.,adj_sorted)
 return adjusted

def dm_hac(y,mbm,mlp,lag=12):
 """Diebold-Mariano approx normal, lags HAC Bartlett; análise exploratória de diferenças de perdas."""
 d=(np.asarray(y)-np.asarray(mbm))**2-(np.asarray(y)-np.asarray(mlp))**2
 d=d[np.isfinite(d)];n=len(d)
 if n<max(8,lag+3):return {'DM_stat':np.nan,'DM_p':np.nan,'DM_n':n,'DM_loss_delta':np.nan}
 centered=d-d.mean();v=np.dot(centered,centered)/n
 for j in range(1,min(lag,n-2)+1):
  gamma=np.dot(centered[j:],centered[:-j])/n
  v+=2*(1-j/(lag+1))*gamma
 if v<=0:return {'DM_stat':np.nan,'DM_p':np.nan,'DM_n':n,'DM_loss_delta':float(np.mean(d))}
 z=float(np.mean(d)/np.sqrt(v/n))
 return {'DM_stat':z,'DM_p':float(2*norm.sf(abs(z))),'DM_n':n,'DM_loss_delta':float(np.mean(d))}

def original_finf_on_development(ex,res,fraction,protocol,conf):
 de=subset(ex,('train','validation'))
 mbm=fit_mbm(ex,de)
 vals=[];kvals=[]
 for seed in CFG.seeds:
  weight=OUT/'modelos'/f'{res}_{fraction.replace(".","")}_{protocol}_seed{seed}.pt'
  state=torch.load(weight,map_location='cpu',weights_only=False)
  model=PhysicalMLP(ex['X'].shape[1],conf['hidden'],conf['dropout'])
  model.load_state_dict(state['model_state'])
  X=transform_ex(ex,state['imputer'],state['scaler'])
  f,k=model_parameters(model,X)
  vals.append(float(f[de].mean()));kvals.append(float(k[de].mean()))
 return mbm,{'Finf':float(np.mean(vals)),'kprime':float(np.mean(kvals))}

def _one_bootstrap(ex,res,fraction,protocol,conf,rep,n_base,te,dev,boot_path):
 rng=np.random.default_rng(20261031+rep*7907+int(res[1:])*997+{'PM1':1,'PM2.5':2,'PM10':3}[fraction]*53+
                           (0 if protocol=='B' else 13))
 ix=dev[mbb_indices(len(dev),CFG.mbb_block,rng)]
 mb=fit_mbm(ex,ix)
 fmbm=np.full(len(ex['t']),mb['Finf']);kmbm=np.full(len(ex['t']),mb['kprime'])
 mbpred,_=evaluate_full_params(ex,(fmbm,kmbm),te,protocol)
 fmeans=[];kmeans=[];preds=[];metrics=[]
 for seed in CFG.seeds:
  model=train_net(ex,ix,np.empty(0,dtype=int),conf,seed,epochs=min(conf['epochs'],CFG.mbb_max_epochs),
                  early_stop=False,store_history=False)
  f,k=model_parameters(model['net'],model['X'])
  fmeans.append(float(np.mean(f[dev])));kmeans.append(float(np.mean(k[dev])))
  pred,mt=evaluate_full_params(ex,(f,k),te,protocol)
  preds.append(pred);metrics.append(mt['RMSE'])
 ensemble_pred=np.mean(preds,axis=0)
 return {'residence':res,'fraction':fraction,'protocol':protocol,'rep':rep,
        'block_size':CFG.mbb_block,'n_fit':len(ix),
        'Finf_MBM':mb['Finf'],'kprime_MBM':mb['kprime'],
        'Finf_MLP':float(np.mean(fmeans)),'kprime_MLP':float(np.mean(kmeans)),
        'RMSE_MBM':mean_metrics(ex['y'][te],mbpred)['RMSE'],
        'RMSE_MLP_ensemble':mean_metrics(ex['y'][te],ensemble_pred)['RMSE'],
        'RMSE_MLP_mean_seeds':float(np.mean(metrics)),
        'Finf_difference':float(mb['Finf']-np.mean(fmeans)),
        'RMSE_difference':float(mean_metrics(ex['y'][te],mbpred)['RMSE']-np.mean(metrics))}

def run_bootstrap_for_case(frame,res,fraction,protocol,conf,target_reps=None):
 ex=make_examples(frame,conf['features'])
 dev=subset(ex,('train','validation'));te=subset(ex,('test',))
 stem=f'{res}_{fraction.replace(".","")}_{protocol}'
 csvpath=OUT/'replicas_mbb'/f'{stem}_paired.csv'
 if csvpath.exists() and not CFG.update_existing:
  cached=pd.read_csv(csvpath)
  records=cached.to_dict('records')
 else:records=[]
 present=set(int(r['rep']) for r in records)
 maxB=int(target_reps or CFG.n_mbb)
 for rep in range(1,maxB+1):
  if rep in present:continue
  row=_one_bootstrap(ex,res,fraction,protocol,conf,rep,maxB,te,dev,csvpath)
  records.append(row);present.add(rep)
  if rep%10==0 or rep==maxB:
   temp=csvpath.with_suffix('.tmp')
   pd.DataFrame(records).sort_values('rep').to_csv(temp,index=False)
   os.replace(temp,csvpath)
 if len(records)<maxB or len({r['rep'] for r in records})!=len(records):raise RuntimeError('Inconsistência nas réplicas MBB')
 return pd.DataFrame(records).sort_values('rep'),ex

B_TABLES=[];STAT_INFERENCE=[]
if CFG.run_mbb:
 for res in CFG.residences:
  for frac in CFG.fractions:
   frame=prepared_case(DATASETS[res],res,frac)
   for protocol in CFG.mbb_protocols:
    js=OUT/'optuna'/f'{res}_{frac.replace(".","")}_{protocol}_selected.json'
    if not js.exists():raise RuntimeError('Treine e selecione modelos antes de MBB')
    conf=json.loads(js.read_text(encoding='utf8'))
    boot,ex=run_bootstrap_for_case(frame,res,frac,protocol,conf)
    mb,hy=original_finf_on_development(ex,res,frac,protocol,conf)
    observed=mb['Finf']-hy['Finf']
    observed_k=mb['kprime']-hy['kprime']
    diff=boot.Finf_difference.to_numpy()
    test=centered_bootstrap_test(diff,observed)
    test_k=centered_bootstrap_test((boot.kprime_MBM-boot.kprime_MLP).to_numpy(),observed_k)
    row={'residence':res,'fraction':frac,'protocol':protocol,
         'Finf_MBM_original':mb['Finf'],'Finf_MLP_original':hy['Finf'],
         'delta_original':observed,**test,
         'delta_k_original':observed_k,**{'k_'+name:value for name,value in test_k.items()}}
    for label,col in [('Finf_MBM','Finf_MBM'),('Finf_MLP','Finf_MLP'),
                      ('kprime_MBM','kprime_MBM'),('kprime_MLP','kprime_MLP'),
                      ('delta_Finf','Finf_difference'),('RMSE_MBM','RMSE_MBM'),
                      ('RMSE_MLP','RMSE_MLP_mean_seeds'),('delta_RMSE','RMSE_difference')]:
     for key,val in sample_summary(boot[col].values).items():row[f'{label}_{key}']=val
    B_TABLES.append(row)
    print('MBB concluído:',res,frac,protocol,len(boot),flush=True)
 # Etapa adaptativa: só casos cujo p inicial pode alcançar algum limiar Holm.
 # Usa novos sorteios (rep novo -> seed diferente); republica CSV atomicamente.
 if CFG.mbb_adaptive and len(B_TABLES)>1 and not CFG.quick_test:
  base_df=pd.DataFrame(B_TABLES)
  for prot in base_df.protocol.unique():
   base_proto=base_df[base_df.protocol==prot].copy()
   m=len(base_proto)
   # Critério conservador de inclusão: p menor que 0,05, sujeito a cruzar algum limiar Holm.
   for _,candidate in base_proto[(base_proto.p_bootstrap<=.05)|(base_proto.k_p_bootstrap<=.05)].iterrows():
    res,frac=candidate.residence,candidate.fraction
    print('MBB adaptativo:',res,frac,prot,'para',CFG.mbb_adaptive_to,'réplicas',flush=True)
    frame=prepared_case(DATASETS[res],res,frac)
    cpath=OUT/'optuna'/f'{res}_{frac.replace(".","")}_{prot}_selected.json'
    conf=json.loads(cpath.read_text(encoding='utf8'))
    boot,ex=run_bootstrap_for_case(frame,res,frac,prot,conf,target_reps=CFG.mbb_adaptive_to)
    observed=float(candidate.delta_original)
    test=centered_bootstrap_test(boot.Finf_difference.to_numpy(),observed)
    test_k=centered_bootstrap_test((boot.kprime_MBM-boot.kprime_MLP).to_numpy(),float(candidate.delta_k_original))
    idx=next(i for i,r in enumerate(B_TABLES) if r['residence']==res and r['fraction']==frac and r['protocol']==prot)
    row=B_TABLES[idx];row.update(test);row.update({'k_'+name:value for name,value in test_k.items()})
    for label,col in [('Finf_MBM','Finf_MBM'),('Finf_MLP','Finf_MLP'),
                      ('kprime_MBM','kprime_MBM'),('kprime_MLP','kprime_MLP'),
                      ('delta_Finf','Finf_difference'),('RMSE_MBM','RMSE_MBM'),
                      ('RMSE_MLP','RMSE_MLP_mean_seeds'),('delta_RMSE','RMSE_difference')]:
     for key,val in sample_summary(boot[col].values).items():row[f'{label}_{key}']=val
 if B_TABLES:
  BOOT_STATS=pd.DataFrame(B_TABLES)
  for prot in BOOT_STATS.protocol.unique():
   ids=BOOT_STATS.protocol==prot
   BOOT_STATS.loc[ids,'p_holm_9']=holm_adjust(BOOT_STATS.loc[ids,'p_bootstrap'].values)
   BOOT_STATS.loc[ids,'k_p_holm_9']=holm_adjust(BOOT_STATS.loc[ids,'k_p_bootstrap'].values)
   # Limites conservadores de precisão MC ajustados em separado.
   BOOT_STATS.loc[ids,'MC_p_upper_holm']=holm_adjust(BOOT_STATS.loc[ids,'MC_p_upper95'].values)
   BOOT_STATS.loc[ids,'k_MC_p_upper_holm']=holm_adjust(BOOT_STATS.loc[ids,'k_MC_p_upper95'].values)
  BOOT_STATS['conclusao_MC']=np.where(BOOT_STATS.MC_p_upper_holm<.05,'detecção estável ao limite MC',
                          np.where(BOOT_STATS.p_holm_9<.05,'nominal; precisão MC insuficiente','não detectado'))
  BOOT_STATS['conclusao_k_MC']=np.where(BOOT_STATS.k_MC_p_upper_holm<.05,'detecção estável ao limite MC',
                          np.where(BOOT_STATS.k_p_holm_9<.05,'nominal; precisão MC insuficiente','não detectado'))
  BOOT_STATS.to_csv(OUT/'tabelas'/'MBB_9casos_Finf_kprime_RMSE_Holm.csv',index=False)
 else:BOOT_STATS=pd.DataFrame()
else:BOOT_STATS=pd.DataFrame()

# Testes de erros preditivos: múltiplos testes de DM/HAC em protocolo e semente.
DM_ROWS=[]
for (res,frac,prot,seed),g in PRED.query("period=='test'").groupby(['residence','fraction','protocol','seed']):
 g=g.sort_values('row_index')
 DM_ROWS.append({'residence':res,'fraction':frac,'protocol':prot,'seed':seed,
                 **dm_hac(g.C_observed_corrected,g.MBM,g.MBM_MLP,CFG.window)})
DM_TABLE=pd.DataFrame(DM_ROWS)
if not DM_TABLE.empty:
 for (prot,seed),idx in DM_TABLE.groupby(['protocol','seed']).groups.items():
  selected=DM_TABLE.loc[idx,'DM_p'].to_numpy()
  valid=np.isfinite(selected);adjusted=np.full(len(selected),np.nan)
  if valid.any():adjusted[valid]=holm_adjust(selected[valid])
  DM_TABLE.loc[idx,'DM_p_holm']=adjusted
 DM_TABLE.to_csv(OUT/'tabelas'/'Diebold_Mariano_HAC_Holm.csv',index=False)


## 5. Figuras 2D e 3D em dois idiomas

O bloco abaixo pertence ao protocolo completo.


In [ ]:
# ==========================================================
# Figuras: PT e EN nas MESMAS FONTES; Matplotlib e SHAP.
# Todas as figuras de F_inf e k_prime são SEPARADAS.
# ==========================================================
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.titlesize':10,
 'axes.labelsize':10,'xtick.labelsize':8,'ytick.labelsize':8,
 'axes.linewidth':0.85,'savefig.facecolor':'white','figure.facecolor':'white'})
FIG_CATALOG=[]

def graph_axes(ax):
 ax.grid(False)
 for spine in ['top','bottom','left','right']:
  ax.spines[spine].set_visible(True)
  ax.spines[spine].set_linewidth(.9)
 ax.tick_params(axis='both',direction='out',length=3,width=.8)

def save_figure(fig,lang,category,slug,caption,source):
 folder=OUT/'figuras'/lang/category
 folder.mkdir(parents=True,exist_ok=True)
 file=folder/f'{slug}.png'
 fig.savefig(file,dpi=CFG.plot_dpi,bbox_inches='tight',pad_inches=.16)
 plt.close(fig)
 FIG_CATALOG.append({'language':lang,'figure_id':slug,'category':category,'file':str(file.relative_to(OUT)),
                     'caption':caption,'data_source':source,'dpi':CFG.plot_dpi})

def panel_xlabel(fig,name):fig.supxlabel(name,x=.5,y=.01)
def panel_ylabel(fig,name):fig.supylabel(name,x=.01)

def paired_test_rows(res,frac,protocol):
 g=PRED[(PRED.residence==res)&(PRED.fraction==frac)&(PRED.protocol==protocol)&(PRED.period=='test')]
 return g

def fig_timeseries_case(res,frac,protocol,lang):
 g=paired_test_rows(res,frac,protocol)
 if g.empty:return
 mat=g.groupby('row_index',sort=True).agg(y=('C_observed_corrected','mean'),mb=('MBM','mean'),
     hy=('MBM_MLP','mean'),hy_sd=('MBM_MLP',lambda v:float(np.std(v,ddof=1)) if len(v)>1 else 0),hy_n=('MBM_MLP','count')).reset_index()
 # Mostrar no máximo 350 observações por painel para não virar linha indiscernível.
 if len(mat)>350:mat=mat.iloc[np.linspace(0,len(mat)-1,350,dtype=int)].copy()
 t=mat.row_index.to_numpy();sd=mat.hy_sd.to_numpy();y=mat.y.to_numpy();mb=mat.mb.to_numpy();hy=mat.hy.to_numpy()
 fig,ax=plt.subplots(figsize=(max(CFG.plot_width_inches,7),3.35),layout='constrained')
 ax.plot(t,y,c=COLORS['observed'],lw=1.12,label=LABEL[lang]['observed'])
 ax.plot(t,mb,c=COLORS['MBM'],lw=1.05,label='MBM')
 ax.plot(t,hy,c=COLORS['Hybrid'],lw=1.05,label='MBM+MLP')
 ax.fill_between(t,hy-sd,hy+sd,color=COLORS['Hybrid'],alpha=.21,label='±1 DP' if lang=='PT' else '±1 SD')
 ci=stats.t.ppf(.975,np.maximum(mat.hy_n.to_numpy()-1,1))*sd/np.sqrt(np.maximum(mat.hy_n.to_numpy(),1))
 ax.fill_between(t,hy-ci,hy+ci,color=COLORS['Hybrid'],alpha=.09,label='IC95% sementes' if lang=='PT' else '95% seed CI')
 ax.set_xlabel(LABEL[lang]['t']);ax.set_ylabel(LABEL[lang]['c']);graph_axes(ax)
 ax.legend(frameon=False,ncol=4,loc='upper center',bbox_to_anchor=(.5,1.19),fontsize=8)
 name=f'{res} · {frac} · '+('Protocolo' if lang=='PT' else 'Protocol')+f' {protocol}'
 ax.set_title(name,pad=27)
 save_figure(fig,lang,'01_temporal',f'{res}_{frac.replace(".","")}_{protocol}_series',name,
             'tabelas/*_predicoes.csv | reentrenamiento três sementes')

def fig_train_test_comparison(res,frac,protocol,lang):
 # Dois períodos distintos; treino mostrado como reajuste in-sample, teste previamente consultado.
 d=PRED[(PRED.residence==res)&(PRED.fraction==frac)&(PRED.protocol==protocol)]
 if d.empty:return
 fig,axs=plt.subplots(1,2,figsize=(10.5,3.55),layout='constrained',sharey=True)
 both=d[d.period.isin(['train','test'])]
 if both.empty:return
 vmax=float(np.nanpercentile(np.r_[both.C_observed_corrected,both.MBM,both.MBM_MLP],99.5)*1.1)
 for ax,period in zip(axs,['train','test']):
  g=d[d.period==period].groupby('row_index',sort=True).agg(y=('C_observed_corrected','mean'),
       mb=('MBM','mean'),hy=('MBM_MLP','mean'),sd=('MBM_MLP',lambda a:np.std(a,ddof=1) if len(a)>1 else 0))
  if len(g)>260:g=g.iloc[np.linspace(0,len(g)-1,260,dtype=int)]
  x=g.index.to_numpy();ax.plot(x,g.y,c=COLORS['observed'],lw=1.05,label=LABEL[lang]['observed'])
  ax.plot(x,g.mb,c=COLORS['MBM'],lw=.95,label='MBM')
  ax.plot(x,g.hy,c=COLORS['Hybrid'],lw=1.05,label='MBM+MLP')
  ax.fill_between(x,g.hy-g.sd,g.hy+g.sd,color=COLORS['Hybrid'],alpha=.18)
  ax.set_ylim(0,max(1,vmax));ax.set_title(LABEL[lang][period]);graph_axes(ax)
 panel_xlabel(fig,LABEL[lang]['t']);panel_ylabel(fig,LABEL[lang]['c'])
 h,l=axs[0].get_legend_handles_labels()
 fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,1.065),ncol=3,frameon=False)
 save_figure(fig,lang,'01_temporal',f'{res}_{frac.replace(".","")}_{protocol}_train_vs_test',
   f'{res} {frac}: training and previously inspected test in {protocol}, same scale',
   'tabelas/previsoes_todas.csv')

def fig_scatter_case(res,protocol,lang):
 fig,axs=plt.subplots(1,3,figsize=(11.4,3.7),layout='constrained',sharex=True,sharey=True)
 rangevals=[]
 for fr in CFG.fractions:
  gg=paired_test_rows(res,fr,protocol)
  if len(gg):rangevals.extend(gg[['C_observed_corrected','MBM','MBM_MLP']].to_numpy().flatten())
 common_upper=max(1.,float(np.nanpercentile(rangevals,99.6)*1.08)) if rangevals else 1.
 for ax,frac in zip(axs,CFG.fractions):
  g=paired_test_rows(res,frac,protocol)
  if g.empty:
   graph_axes(ax);ax.set_visible(False);continue
  # média das três sementes por instante
  mat=g.groupby('row_index',sort=True).agg(y=('C_observed_corrected','mean'),mb=('MBM','mean'),hy=('MBM_MLP','mean'))
  xx=mat.y.to_numpy();yy=mat.mb.to_numpy();zz=mat.hy.to_numpy()
  upper=common_upper
  ax.scatter(xx,yy,c=COLORS['MBM'],s=9,alpha=.42,edgecolors='none',label='MBM',rasterized=True)
  ax.scatter(xx,zz,c=COLORS['Hybrid'],s=9,alpha=.42,edgecolors='none',label='MBM+MLP',rasterized=True)
  ax.plot([0,upper],[0,upper],ls=':',c='.25',lw=1,label='1:1')
  ax.set(xlim=(0,upper),ylim=(0,upper));ax.set_aspect('equal',adjustable='box');ax.set_title(frac);graph_axes(ax)
 panel_xlabel(fig,LABEL[lang]['x']);panel_ylabel(fig,LABEL[lang]['y'])
 handles,labels=axs[0].get_legend_handles_labels()
 fig.legend(handles,labels,loc='upper center',ncol=3,frameon=False,bbox_to_anchor=(.5,1.065))
 save_figure(fig,lang,'02_comparacao',f'{res}_P{protocol}_scatter',f'{res} {protocol} — test scatter plots','tabelas/previsoes_todas.csv')

def fig_rmse_case(res,lang):
 fig,axs=plt.subplots(1,2,figsize=(8.8,3.65),layout='constrained',sharey=True)
 for ax,protocol in zip(axs,CFG.protocols):
  g=MET[(MET.residence==res)&(MET.period=='test')&(MET.protocol==protocol)]
  if g.empty:continue
  xs=np.arange(len(CFG.fractions));w=.35
  for xmod,name,color in [(0,'MBM',COLORS['MBM']),(1,'MBM_MLP',COLORS['Hybrid'])]:
   arr=[];bars=[]
   for frac in CFG.fractions:
    vals=g[(g.fraction==frac)&(g.model==name)].RMSE.to_numpy()
    arr.append(np.mean(vals) if len(vals) else np.nan)
    bars.append(np.std(vals,ddof=1) if len(vals)>1 else 0)
   bars=np.asarray(bars);nseeds=len(CFG.seeds)
   ci=stats.t.ppf(.975,max(nseeds-1,1))*bars/np.sqrt(max(nseeds,1))
   xc=xs+(xmod-.5)*w
   ax.bar(xc,arr,width=w,yerr=bars,capsize=3,color=color,alpha=.95,
          label='MBM' if xmod==0 else 'MBM+MLP',error_kw={'lw':1,'capthick':1})
   if xmod==1 and nseeds>1:ax.errorbar(xc,arr,yerr=ci,fmt='none',color='#262626',elinewidth=.9,capsize=5)
  ax.set_xticks(xs,CFG.fractions);ax.set_title(('Protocolo' if lang=='PT' else 'Protocol')+' '+protocol)
  graph_axes(ax)
 panel_ylabel(fig,LABEL[lang]['rmse']);h,l=axs[0].get_legend_handles_labels()
 fig.legend(h,l,ncol=2,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.065))
 save_figure(fig,lang,'02_comparacao',f'{res}_rmse_sd',f'{res}: RMSE mean ± between-seed SD','tabelas/metricas_todas.csv')

def fig_learning(res,frac,protocol,lang):
 p=OUT/'optuna'/f'{res}_{frac.replace(".","")}_{protocol}_learning_curves.csv'
 if not p.exists():return
 d=pd.read_csv(p);fig,ax=plt.subplots(figsize=(6.7,3.55),layout='constrained')
 for col,color,label in [('train_RMSE_A',COLORS['MBM'],'Treino 1 passo' if lang=='PT' else 'One-step training'),
                         ('validation_RMSE',COLORS['validation'],'Validação' if lang=='PT' else 'Validation')]:
  grp=d.groupby('epoch')[col];mean=grp.mean();std=grp.std().fillna(0)
  # Histórico pode conter épocas com apenas uma semente: não interpretar faixa como IC.
  t=mean.index.to_numpy();y=mean.to_numpy();s=std.to_numpy()
  ax.plot(t,y,c=color,label=label,lw=1.6)
  ax.fill_between(t,y-s,y+s,color=color,alpha=.16)
 ax.set_xlabel('Época' if lang=='PT' else 'Epoch');ax.set_ylabel(LABEL[lang]['rmse'])
 graph_axes(ax);ax.legend(frameon=False,ncol=2,fontsize=8)
 ax.set_title(f'{res} · {frac} · {protocol}: '+('curvas de aprendizagem' if lang=='PT' else 'learning curves'))
 save_figure(fig,lang,'05_desenv',f'{res}_{frac.replace(".","")}_{protocol}_learning',f'{res} · {frac}: train and validation learning curves',str(p.relative_to(OUT)))

def fig_parameter_case(res,frac,protocol,parameter,lang):
 g=paired_test_rows(res,frac,protocol)
 col='Finf_MLP' if parameter=='F' else 'kprime_MLP'
 base='Finf_MBM' if parameter=='F' else 'kprime_MBM'
 if g.empty:return
 agg=g.groupby('row_index',sort=True).agg(mean=(col,'mean'),lo=(col,'min'),hi=(col,'max'),sd=(col,lambda x:np.std(x,ddof=1) if len(x)>1 else 0),n=(col,'count'),ref=(base,'mean')).reset_index()
 if len(agg)>500:agg=agg.iloc[np.linspace(0,len(agg)-1,500,dtype=int)]
 x=agg.row_index.to_numpy();a=agg['mean'].to_numpy();lo=agg.lo.to_numpy();hi=agg.hi.to_numpy()
 fig,ax=plt.subplots(figsize=(7.1,3.35),layout='constrained')
 ax.fill_between(x,lo,hi,color=COLORS['Hybrid'],alpha=.09,label='min–max')
 sd=agg.sd.to_numpy();n=agg.n.to_numpy();ci=stats.t.ppf(.975,np.maximum(n-1,1))*sd/np.sqrt(np.maximum(n,1))
 ax.fill_between(x,a-ci,a+ci,color=COLORS['Hybrid'],alpha=.12,label='IC95% sementes' if lang=='PT' else '95% seed CI')
 ax.fill_between(x,a-sd,a+sd,color=COLORS['Hybrid'],alpha=.22,label='±1 DP' if lang=='PT' else '±1 SD')
 ax.plot(x,a,c=COLORS['Hybrid'],lw=1.35,label='MBM+MLP')
 ax.axhline(float(agg.ref.mean()),c=COLORS['MBM'],lw=1.25,linestyle='--',label='MBM')
 if parameter=='F':ax.set_ylim(0,1.02)
 ax.set_xlabel(LABEL[lang]['t']);ax.set_ylabel(LABEL[lang]['f'] if parameter=='F' else LABEL[lang]['k'])
 graph_axes(ax);ax.legend(frameon=False,ncol=3,loc='upper center',bbox_to_anchor=(.5,1.27),fontsize=7)
 ax.set_title(f'{res} · {frac} · {protocol}: '+('F_inf(t)' if parameter=='F' else "k′(t)"),pad=33)
 save_figure(fig,lang,'03_parametros',f'{res}_{frac.replace(".","")}_{protocol}_{parameter}',
             f'{res} · {frac}: dynamic '+parameter+' parameter with between-seed min–max','tabelas/previsoes_todas.csv')

def fig_bootstrap_case(res,frac,par,lang):
 if BOOT_STATS.empty:return
 subset=BOOT_STATS[(BOOT_STATS.residence==res)&(BOOT_STATS.fraction==frac)&(BOOT_STATS.protocol=='B')]
 if subset.empty:return
 row=subset.iloc[0];pr='Finf' if par=='F' else 'kprime'
 names=['MBM','MBM+MLP'];xs=[0,1];colors=[COLORS['MBM'],COLORS['Hybrid']]
 fig,ax=plt.subplots(figsize=(4.25,3.8),layout='constrained')
 for i,tag in enumerate(('MBM','MLP')):
  pref=f'{pr}_{tag}';mu=row.get(pref+'_mean',np.nan)
  lo=row.get(pref+'_CI95_lower',np.nan);hi=row.get(pref+'_CI95_upper',np.nan)
  sd=float(row.get(pref+'_SD',np.nan))
  ax.errorbar(i,mu,yerr=[[mu-lo],[hi-mu]],fmt='o',color=colors[i],markersize=6,
              capsize=6,elinewidth=1.3,capthick=1.4)
  if np.isfinite(sd):ax.plot([i,i],[mu-sd,mu+sd],color=colors[i],lw=5,alpha=.4,solid_capstyle='butt')
 ax.set_xticks(xs,names);ax.set_xlim(-.5,1.5)
 ax.set_ylabel(LABEL[lang]['f'] if par=='F' else LABEL[lang]['k'])
 if par=='F':ax.set_ylim(0,1.04)
 ax.set_title(f'{res} · {frac} · '+('IC95% MBB' if lang=='PT' else '95% MBB CI'))
 graph_axes(ax)
 save_figure(fig,lang,'04_incerteza',f'{res}_{frac.replace(".","")}_MBB_{par}',
             f'{res}: paired MBB mean and 95% percentile CI',
             'tabelas/MBB_9casos_Finf_kprime_RMSE_Holm.csv')

def fig_heatmap_group(group,lang):
 rows=[];names=[]
 for res in group:
  vals=[]
  for frac in CFG.fractions:
   g=MET[(MET.residence==res)&(MET.fraction==frac)&(MET.protocol=='B')&
         (MET.period=='test')]
   if len(g):
    b=g[g.model=='MBM'].RMSE.mean();h=g[g.model=='MBM_MLP'].RMSE.mean()
    vals.append(100*(b-h)/b if b>0 else np.nan)
   else:vals.append(np.nan)
  rows.append(vals);names.append(res)
 array=np.asarray(rows,dtype=float);M=np.nanmax(np.abs(array));M=max(M,1.)
 fig,ax=plt.subplots(figsize=(5.5,max(2.5,1.1+len(group)*.78)),layout='constrained')
 cmap=plt.get_cmap('RdBu');im=ax.imshow(array,cmap=cmap,vmin=-M,vmax=M,aspect='auto')
 ax.set_xticks(range(len(CFG.fractions)),CFG.fractions);ax.set_yticks(range(len(group)),names)
 for i in range(len(group)):
  for j in range(len(CFG.fractions)):
   ax.text(j,i,f'{array[i,j]:+.1f}%',ha='center',va='center',color='white' if abs(array[i,j])>.55*M else 'black',fontsize=10)
 graph_axes(ax)
 cbar=fig.colorbar(im,ax=ax,fraction=.04,pad=.03)
 cbar.set_label('Melhoria relativa de RMSE (%)' if lang=='PT' else 'Relative RMSE improvement (%)')
 save_figure(fig,lang,'02_comparacao',f'{"_".join(group)}_heatmap',
             'Relative RMSE gains in protocol B, grouped residences','tabelas/metricas_todas.csv')

def fig_3d_case(res,frac,lang,par):
 # Diagramas 3D distintos para F_inf(t) e k′(t); sem colocá-los no mesmo painel.
 g=paired_test_rows(res,frac,'B')
 if g.empty:return
 d=g.groupby('row_index',sort=True).agg(cin=('C_observed_corrected','mean'),
    cout=('Cout','mean'),f=('Finf_MLP','mean'),k=('kprime_MLP','mean'))
 if len(d)>350:d=d.iloc[np.linspace(0,len(d)-1,350,dtype=int)]
 fig=plt.figure(figsize=(6.25,4.8),layout='constrained');ax=fig.add_subplot(111,projection='3d')
 z=d.f if par=='F' else d.k
 points=ax.scatter(d.cout,d.cin,z,c=d.index,cmap='viridis',s=8,alpha=.65,rasterized=True)
 ax.set_xlabel('Cout (µg m⁻³)',labelpad=9);ax.set_ylabel('Cin (µg m⁻³)',labelpad=9)
 ax.set_zlabel(LABEL[lang]['f'] if par=='F' else LABEL[lang]['k'],labelpad=8)
 ax.view_init(elev=20,azim=-58)
 ax.set_title(f'{res} · {frac} · '+('F_inf(t)' if par=='F' else "k′(t)"),pad=12)
 for axis in (ax.xaxis,ax.yaxis,ax.zaxis):
  try:axis.pane.fill=False
  except Exception:pass
 ax.grid(False)
 fig.colorbar(points,ax=ax,pad=.12,shrink=.65,label=LABEL[lang]['t'])
 save_figure(fig,lang,'06_3D',f'{res}_{frac.replace(".","")}_{par}_3D',
             f'{res} {frac}: 3D exposure and dynamic {par} parameter','tabelas/previsoes_todas.csv')

# Catálogo de imagens e legendas são gerados em cada idioma ao fim da execução.
if CFG.run_figures:
 for lang in ('PT','EN'):
  for res in CFG.residences:
   for frac in CFG.fractions:
    for prot in CFG.protocols:
     fig_timeseries_case(res,frac,prot,lang)
     fig_train_test_comparison(res,frac,prot,lang)
     fig_learning(res,frac,prot,lang)
     for par in ('F','K'):fig_parameter_case(res,frac,prot,par,lang)
    for par in ('F','K'):
     fig_bootstrap_case(res,frac,par,lang)
     if CFG.run_3d:fig_3d_case(res,frac,lang,par)
   for prot in CFG.protocols:fig_scatter_case(res,prot,lang)
   fig_rmse_case(res,lang)
  for group in (['R1','R2','R3','R4'],['R5','R6','R7'],['R8']):
   selected=[r for r in group if r in CFG.residences]
   if selected:fig_heatmap_group(selected,lang)
 pd.DataFrame(FIG_CATALOG).to_csv(OUT/'tabelas'/'catalogo_figuras_bilingue.csv',index=False)
 print('Figuras geradas:',len(FIG_CATALOG))


## 6. Ablação, SHAP, preditores, catálogos e insights

O bloco abaixo pertence ao protocolo completo.


In [ ]:
# ==========================================================
# Ablação controlada (sem retuning), SHAP para F_inf e k′,
# relações com preditores e relatório de interpretação.
# ==========================================================
ABLATION_RESULTS=[];SHAP_RESULTS=[];CORR_RESULTS=[];TABLE_CATALOG=[]

def fig_bivariate_finf(res,frac,protocol,lang,frame,features):
 fdata=PRED[(PRED.residence==res)&(PRED.fraction==frac)&(PRED.protocol==protocol)&(PRED.period=='test')]
 if fdata.empty:return
 means=fdata.groupby('row_index',sort=True)['Finf_MLP'].mean().rename('F').reset_index()
 both=means.merge(frame.reset_index(drop=True)[['row_index']+features],on='row_index',how='left')
 keep=[v for v in features if v in ('Cout','Tin','Tout','URin','URout','ws','precip_cat','hour_sin')]
 if not keep:return
 take=keep[:4]
 fig,axs=plt.subplots(1,len(take),figsize=(3.3*len(take),3.55),layout='constrained')
 axs=np.atleast_1d(axs)
 for ax,f in zip(axs,take):
  x=both[f].to_numpy(float);y=both.F.to_numpy(float);mask=np.isfinite(x)&np.isfinite(y)
  ax.scatter(x[mask],y[mask],s=8,alpha=.26,c=COLORS['Hybrid'],edgecolor='none',rasterized=True)
  if mask.sum()>20 and len(np.unique(x[mask]))>6:
   # Agrupamento de quantis apenas descritivo, não PDP causal.
   bins=np.unique(np.nanquantile(x[mask],np.linspace(0,1,13)))
   if len(bins)>3:
    cuts=np.digitize(x[mask],bins[1:-1]);xx=[];yy=[]
    for b in np.unique(cuts):
     xx.append(np.median(x[mask][cuts==b]));yy.append(np.median(y[mask][cuts==b]))
    ax.plot(xx,yy,c=COLORS['observed'],lw=1.6,label='mediana por faixa' if lang=='PT' else 'binned median')
  ax.set_ylim(0,1.02);ax.set_xlabel(f);ax.set_title(f)
  graph_axes(ax)
 panel_ylabel(fig,LABEL[lang]['f'])
 save_figure(fig,lang,'07_preditores',f'{res}_{frac.replace(".","")}_{protocol}_F_predictors',
             f'{res} {frac}: associations of F_inf(t) with predictors (descriptive)',
             'tabelas/previsoes_todas.csv; dados_processados/')
 for f in take:
  q=both[[f,'F']].dropna()
  if len(q)>6 and q[f].nunique()>3:
   sp=stats.spearmanr(q[f],q.F)
   CORR_RESULTS.append({'residence':res,'fraction':frac,'protocol':protocol,
                        'feature':f,'Spearman_rho':sp.statistic,'p_nominal':sp.pvalue,
                        'n':len(q),'exploratory':'yes; serial dependence unaccounted'})

def run_ablation_case(res,frac,protocol,frame,conf):
 from copy import deepcopy
 all_features=list(conf['features'])
 if len(all_features)<2:return
 base_ex=make_examples(frame,all_features)
 tr=subset(base_ex,['train']);va=subset(base_ex,['validation'])
 orig=[]
 for seed in CFG.seeds:
  m=train_net(base_ex,tr,va,conf,seed,epochs=conf['epochs'],early_stop=False,store_history=False)
  f,k=model_parameters(m['net'],m['X'])
  pred,_=evaluate_full_params(base_ex,(f,k),va,protocol)
  orig.append(mean_metrics(base_ex['y'][va],pred)['RMSE'])
 for feature in all_features:
  if feature=='Cout':continue # coluna externa é estrutural na equação
  leftover=[f for f in all_features if f!=feature]
  ex=make_examples(frame,leftover)
  itr=subset(ex,['train']);iva=subset(ex,['validation'])
  vals=[]
  for seed in CFG.seeds:
   fitted=train_net(ex,itr,iva,conf,seed,epochs=conf['epochs'],early_stop=False,store_history=False)
   f,k=model_parameters(fitted['net'],fitted['X'])
   pred,_=evaluate_full_params(ex,(f,k),iva,protocol)
   vals.append(mean_metrics(ex['y'][iva],pred)['RMSE'])
  for seed,base,drop in zip(CFG.seeds,orig,vals):
   ABLATION_RESULTS.append({'residence':res,'fraction':frac,'protocol':protocol,
      'seed':seed,'removed_feature':feature,'base_val_RMSE':base,
      'removed_val_RMSE':drop,'delta_RMSE':drop-base,
      'percentage':100*(drop-base)/max(base,1e-8)})

def run_shap_case(res,frac,protocol,conf,ex):
 if not SHAP_AVAILABLE:return
 stem=f'{res}_{frac.replace(".","")}_{protocol}'
 features=conf['features'];num_lags=CFG.window
 train=subset(ex,['train']);te=subset(ex,['test'])
 if not len(train) or not len(te):return
 for seed in CFG.seeds:
  checkpoint=OUT/'modelos'/f'{stem}_seed{seed}.pt'
  if not checkpoint.exists():continue
  state=torch.load(checkpoint,map_location='cpu',weights_only=False)
  model=PhysicalMLP(ex['X'].shape[1],conf['hidden'],conf['dropout'])
  model.load_state_dict(state['model_state']);model.eval()
  X=transform_ex(ex,state['imputer'],state['scaler'])
  rng=np.random.default_rng(CFG.shap_seed+int(seed))
  bg=X[rng.choice(train,size=min(len(train),CFG.shap_background),replace=False)]
  samples=X[rng.choice(te,size=min(len(te),CFG.shap_max_samples),replace=False)]
  for field,pname in [('F','F_inf'),('K','k_prime')]:
   def predict(xx):
    ff,kk=model_parameters(model,np.asarray(xx,dtype=np.float32))
    return ff if field=='F' else kk
   try:
    masker=shap.maskers.Independent(bg)
    expl=shap.Explainer(predict,masker,algorithm='permutation')
    explanation=expl(samples,max_evals=max(2*X.shape[1]+1,50),silent=True)
    sv=np.asarray(explanation.values)
    weights=np.nanmean(np.abs(sv.reshape(len(samples),num_lags,len(features))),axis=(0,1))
    for ft,v in zip(features,weights):
     SHAP_RESULTS.append({'residence':res,'fraction':frac,'protocol':protocol,
       'parameter':pname,'feature':ft,'mean_abs_SHAP':float(v),
       'n_test_samples':len(samples),'background_n':len(bg),'seed':seed,
       'interpretation':'association within neural estimator; not causal'})
   except Exception as e:
    with (OUT/'auditoria'/'shap_errors.txt').open('a',encoding='utf8') as fo:
     fo.write(f'{res},{frac},{protocol},{field},{seed}: {e}\n')


def fig_feature_bars(lang):
 if not SHAP_RESULTS:return
 d=pd.DataFrame(SHAP_RESULTS)
 for (res,frac,prot,par),g in d.groupby(['residence','fraction','protocol','parameter']):
  agg=g.groupby('feature').mean_abs_SHAP.agg(['mean','std','count'])
  agg=agg.sort_values('mean').tail(12)
  fig,ax=plt.subplots(figsize=(6.5,4.05),layout='constrained')
  y=np.arange(len(agg));mean=agg['mean'].to_numpy();sd=agg['std'].fillna(0).to_numpy();n=agg['count'].to_numpy()
  ci=stats.t.ppf(.975,np.maximum(n-1,1))*sd/np.sqrt(np.maximum(n,1))
  ax.barh(y,mean,xerr=sd,color=COLORS['Hybrid'],capsize=3,
          error_kw={'lw':1.0,'capthick':1.0})
  ax.errorbar(mean,y,xerr=ci,fmt='none',color=COLORS['observed'],elinewidth=.85,capsize=5)
  ax.set_yticks(y,agg.index)
  ax.set_xlabel(('SHAP médio absoluto ± DP; traço: IC95%' if lang=='PT' else
                 'Mean absolute SHAP ± SD; outer caps: 95% CI')+' — '+par)
  ax.set_title(f'{res} · {frac} · {prot} · {par}')
  graph_axes(ax)
  save_figure(fig,lang,'07_preditores',f'{res}_{frac.replace(".","")}_{prot}_{par}_SHAP',
              f'SHAP mean ± SD [t-based 95% CI], three seeds, dynamic {par}',
              'tabelas/SHAP_parametros.csv')


def fig_ablation_bars(lang):
 if not ABLATION_RESULTS:return
 d=pd.DataFrame(ABLATION_RESULTS)
 for (res,frac,prot),g in d.groupby(['residence','fraction','protocol']):
  agg=g.groupby('removed_feature').percentage.agg(['mean','std','count']).sort_values('mean')
  fig,ax=plt.subplots(figsize=(6.4,3.5),layout='constrained')
  xs=np.arange(len(agg));colors=[COLORS['Hybrid'] if v>=0 else '#BA7563' for v in agg['mean']]
  sd=agg['std'].fillna(0).to_numpy();mu=agg['mean'].to_numpy();n=agg['count'].to_numpy()
  ci=stats.t.ppf(.975,np.maximum(n-1,1))*sd/np.sqrt(np.maximum(n,1))
  ax.barh(xs,mu,xerr=sd,color=colors,capsize=3)
  ax.errorbar(mu,xs,xerr=ci,fmt='none',color='#292929',capsize=5,elinewidth=.9)
  ax.set_yticks(xs,agg.index)
  ax.axvline(0,color='.3',lw=.85)
  ax.set_xlabel('ΔRMSE (%) ± DP; traço: IC95%' if lang=='PT' else 'ΔRMSE (%) ± SD; outer caps: 95% CI')
  graph_axes(ax)
  ax.set_title(f'{res} · {frac} · {prot} · '+('ablação' if lang=='PT' else 'ablation'))
  save_figure(fig,lang,'05_desenv',f'{res}_{frac.replace(".","")}_{prot}_ablation',
              f'Ablation study, mean ± between-seed SD; fixed architecture','tabelas/ablacao_3_sementes.csv')

if CFG.run_ablation or CFG.run_shap or CFG.run_figures:
 for res in CFG.residences:
  for frac in CFG.fractions:
   frame=prepared_case(DATASETS[res],res,frac)
   for prot in CFG.protocols:
    path=OUT/'optuna'/f'{res}_{frac.replace(".","")}_{prot}_selected.json'
    if not path.exists():continue
    conf=json.loads(path.read_text(encoding='utf8'))
    ex=make_examples(frame,conf['features'])
    if CFG.run_ablation:run_ablation_case(res,frac,prot,frame,conf)
    if CFG.run_shap:run_shap_case(res,frac,prot,conf,ex)
    if CFG.run_figures:
     for lang in ('PT','EN'):
      fig_bivariate_finf(res,frac,prot,lang,frame,conf['features'])
if ABLATION_RESULTS:
 _ab=pd.DataFrame(ABLATION_RESULTS)
 _ab.to_csv(OUT/'tabelas'/'ablacao_3_sementes.csv',index=False)
 summaries=[]
 for keys,g in _ab.groupby(['residence','fraction','protocol','removed_feature']):
  a=np.asarray(g['percentage'],float);n=len(a);m=a.mean();sd=np.std(a,ddof=1) if n>1 else np.nan
  ci=stats.t.ppf(.975,n-1)*sd/np.sqrt(n) if n>1 else np.nan
  summaries.append(dict(zip(['residence','fraction','protocol','removed_feature'],keys)) |
    {'mean_change_pct':m,'SD':sd,'CI95_lower':m-ci,'CI95_upper':m+ci,'n_seeds':n})
 pd.DataFrame(summaries).to_csv(OUT/'tabelas'/'ablacao_media_DP_IC95.csv',index=False)
if SHAP_RESULTS:
 pd.DataFrame(SHAP_RESULTS).to_csv(OUT/'tabelas'/'SHAP_parametros.csv',index=False)
 _shap=pd.DataFrame(SHAP_RESULTS)
 _shap_summary=[]
 for keys,gg in _shap.groupby(['residence','fraction','protocol','parameter','feature']):
  z=np.asarray(gg['mean_abs_SHAP'],float);n=len(z);mean=z.mean();sd=np.std(z,ddof=1) if n>1 else np.nan
  err=stats.t.ppf(.975,n-1)*sd/np.sqrt(n) if n>1 else np.nan
  _shap_summary.append(dict(zip(['residence','fraction','protocol','parameter','feature'],keys)) |
    {'mean':mean,'SD':sd,'CI95_lower':mean-err,'CI95_upper':mean+err,'n_seeds':n})
 pd.DataFrame(_shap_summary).to_csv(OUT/'tabelas'/'SHAP_media_DP_IC95.csv',index=False)
if CORR_RESULTS:pd.DataFrame(CORR_RESULTS).to_csv(OUT/'tabelas'/'correlacoes_Finf_preditores.csv',index=False)
if CFG.run_figures:
 for lang in ('PT','EN'):
  fig_feature_bars(lang);fig_ablation_bars(lang)

# === Resumos reproduzíveis com DP e IC95% identificados ===
def t_summary(v):
 v=np.asarray(v,dtype=float);v=v[np.isfinite(v)];n=len(v)
 if not n:return {'mean':np.nan,'SD':np.nan,'CI95_lower':np.nan,'CI95_upper':np.nan,'n':0}
 mu=float(np.mean(v));sd=float(np.std(v,ddof=1)) if n>1 else np.nan
 margin=stats.t.ppf(.975,n-1)*sd/np.sqrt(n) if n>1 else np.nan
 return {'mean':mu,'SD':sd,'CI95_lower':mu-margin,'CI95_upper':mu+margin,'n':n,
         'CI_method':'Student-t across independent seeds; n='+str(n)}
metrics_by_seed=[]
for (res,frac,prot,period,model),g in MET.groupby(['residence','fraction','protocol','period','model']):
 line={'residence':res,'fraction':frac,'protocol':prot,'period':period,'model':model}
 for metric in ['RMSE','MAE','R2','NRMSE','bias']:
  line.update({metric+'_'+k:v for k,v in t_summary(g[metric]).items()})
 metrics_by_seed.append(line)
pd.DataFrame(metrics_by_seed).to_csv(OUT/'tabelas'/'metricas_media_DP_IC95_sementes.csv',index=False)

if FIG_CATALOG:
 pd.DataFrame(FIG_CATALOG).to_csv(OUT/'tabelas'/'catalogo_figuras_bilingue.csv',index=False)

for p in sorted((OUT/'tabelas').glob('*.csv')):
 TABLE_CATALOG.append({'table':p.name,'source':'gerada pelo pipeline após preparação temporal; confira os métodos em README',
                       'description_PT':'Resultados por residência e fração, com método de incerteza indicado no cabeçalho.',
                       'description_EN':'Residence- and fraction-specific results; check the uncertainty method in column headers.'})
pd.DataFrame(TABLE_CATALOG).to_csv(OUT/'tabelas'/'catalogo_tabelas.csv',index=False)

# Insights com rótulos quantitativos, não interpretações causais automáticas.
def insights_language(lang):
 pt=lang=='PT';lines=[]
 lines.append('# '+('Síntese exploratória auditável dos resultados' if pt else 'Auditable exploratory results overview'))
 lines.append('')
 lines.append(('Todos os números são calculados nesta execução. As frases são hipóteses interpretativas, não prova causal.' if pt else
  'All figures are computed by this run. Statements provide interpretative hypotheses, not causal proof.'))
 test=MET[MET.period=='test']
 for res in CFG.residences:
  lines.append('\n## '+res)
  for frac in CFG.fractions:
   q=test[(test.residence==res)&(test.fraction==frac)&(test.protocol=='B')]
   if q.empty:continue
   mb=q[q.model=='MBM'].RMSE.mean();hy=q[q.model=='MBM_MLP'].RMSE.mean()
   diff=100*(mb-hy)/mb if mb>0 else np.nan
   statement=(f'{frac}: RMSE MBM={mb:.3f}; MBM+MLP={hy:.3f} µg/m³; variação relativa={diff:+.1f}%. '
      + ('Híbrido com menor RMSE; avaliar IC e DM antes de afirmar superioridade.' if diff>0 else
         'MBM com menor RMSE; maior complexidade não se traduziu em benefício observado.')) if pt else (
       f'{frac}: MBM RMSE={mb:.3f}; MBM+MLP RMSE={hy:.3f} µg/m³; relative gain={diff:+.1f}%. '
       +('Hybrid has lower RMSE; inspect CIs and DM before claiming superiority.' if diff>0 else
         'MBM has lower RMSE; complexity did not improve this observed test.'))
   lines.append('- '+statement)
   if not BOOT_STATS.empty:
    bb=BOOT_STATS[(BOOT_STATS.residence==res)&(BOOT_STATS.fraction==frac)&(BOOT_STATS.protocol=='B')]
    if len(bb):
     rr=bb.iloc[0];lines.append(f'  - ΔF_inf={rr["delta_original"]:+.4f}; '
       f'p_boot={rr["p_bootstrap"]:.4g}, p_Holm={rr["p_holm_9"]:.4g}; '
       f'{rr["conclusao_MC"]}.')
 lines.append('\n'+('## Limitações obrigatórias' if pt else '## Required limitations'))
 lines.append('- '+('O teste temporal foi consultado anteriormente; resultados não são confirmatórios intocados.' if pt else
                       'The temporal test period was previously inspected; findings are not fully untouched confirmatory tests.'))
 lines.append('- '+('IC das três sementes é t aproximado; IC do MBB é percentílico de reamostragens; não são equivalentes.' if pt else
                       'Three-seed t-intervals differ from percentile MBB confidence intervals.'))
 lines.append('- '+('A censura original v9 inclui fallback não causal dentro de cada partição; consultar auditoria por ponto.' if pt else
                       'Source v9 censoring includes a non-causal fallback within each split; inspect per-point audit.'))
 lines.append('- '+('Parâmetros do híbrido são efetivos; previsões próximas não garantem identificabilidade física.' if pt else
                       'Dynamic parameters are effective model parameters, not independently measured infiltration.'))
 lines.append('- '+('Bootstrap nos exemplos temporais não equivale a repetir aquisição e classificação/censura em todas as réplicas.' if pt else
                       'Example-level block bootstrap does not repeat data acquisition and event classification in every replicate.'))
 return '\n'.join(lines)
for lang in ('PT','EN'):
 (OUT/'insights'/f'INSIGHTS_{lang}.md').write_text(insights_language(lang),encoding='utf8')
print('Catálogos, ablação, SHAP, associações e insights finalizados.')


## Gráficos adicionais: dispersão de treino e teste e densidades bootstrap de Finf
O protocolo B possui bootstrap conforme a configuração original. Os pontos do híbrido são médias das três sementes.

In [ ]:
# Gráficos adicionais: médias das três sementes por instante, com regressões separadas.
from scipy.stats import gaussian_kde
for res in CFG.residences:
 for fr in CFG.fractions:
  for prot in CFG.protocols:
   d=PRED[(PRED.residence==res)&(PRED.fraction==fr)&(PRED.protocol==prot)&PRED.period.isin(['train','test'])]
   if d.empty: continue
   g=d.groupby(['period','row_index'],as_index=False).agg(observed=('C_observed_corrected','mean'), MBM=('MBM','mean'), MBM_MLP=('MBM_MLP','mean'))
   for model,col in [('MBM','MBM'),('MBM+MLP','MBM_MLP')]:
    fig,ax=plt.subplots(figsize=(5.2,5.2),layout='constrained')
    allv=np.concatenate([g.observed.to_numpy(),g[col].to_numpy()]);allv=allv[np.isfinite(allv)]
    lo=min(0,float(allv.min()));hi=float(allv.max());pad=(hi-lo)*.04 or 1;lo-=pad;hi+=pad
    ax.plot([lo,hi],[lo,hi],color='black',ls='--',lw=1,label='1:1')
    for period,color,label in [('train','#E17B2D','Training'),('test','#21825B','Test')]:
     h=g[g.period==period];x=h.observed.to_numpy();y=h[col].to_numpy();ok=np.isfinite(x)&np.isfinite(y);x=x[ok];y=y[ok]
     ax.scatter(x,y,s=8,alpha=.48,color=color,label=label,edgecolors='none')
     if len(x)>1 and np.ptp(x)>0:
      slope,intercept=np.polyfit(x,y,1);xx=np.array([x.min(),x.max()]);ax.plot(xx,slope*xx+intercept,color=color,lw=1.2,label=label+' regression')
    ax.set(xlim=(lo,hi),ylim=(lo,hi),xlabel='Measured indoor concentration (µg m$^{-3}$)',ylabel='Modelled indoor concentration (µg m$^{-3}$)',title=f'{res} · {fr} · {model} · protocol {prot}')
    ax.set_aspect('equal',adjustable='box');ax.legend(frameon=False,fontsize=7);graph_axes(ax)
    save_figure(fig,'EN','08_scatter_train_test',f'{res}_{fr.replace(".","")}_{prot}_{col}_scatter_train_test','Train and test: observed versus predicted; regression per period.','tabelas/previsoes_todas.csv')
  # Os ICs e curvas representam réplicas MBB do parâmetro, não variabilidade temporal.
  for prot in CFG.mbb_protocols:
   p=OUT/'replicas_mbb'/f'{res}_{fr.replace(".","")}_{prot}_paired.csv'
   if not p.exists(): continue
   b=pd.read_csv(p);fig,ax=plt.subplots(figsize=(6.4,4),layout='constrained')
   for col,color,label in [('Finf_MBM','#2866A7','MBM'),('Finf_MLP','#21825B','MBM+MLP')]:
    v=b[col].to_numpy(float);v=v[np.isfinite(v)]
    if len(v)<2:continue
    lo,hi=np.quantile(v,[.025,.975]);xs=np.linspace(0,1,500)
    if np.std(v)>1e-12:
     y=gaussian_kde(v)(xs);ax.plot(xs,y,color=color,lw=1.6,label=f'{label}: 95% CI [{lo:.3f}, {hi:.3f}]');ax.fill_between(xs,0,y,where=(xs>=lo)&(xs<=hi),color=color,alpha=.13)
    else:ax.axvline(v[0],color=color,label=f'{label}: degenerate distribution')
    ax.axvline(lo,color=color,ls=':',lw=.8);ax.axvline(hi,color=color,ls=':',lw=.8)
   ax.set(xlim=(0,1),xlabel='$F_{\mathrm{inf}}$',ylabel='Bootstrap density',title=f'{res} · {fr} · protocol {prot}')
   graph_axes(ax);ax.legend(frameon=False,fontsize=8)
   save_figure(fig,'EN','09_density_Finf',f'{res}_{fr.replace(".","")}_{prot}_Finf_density','Paired MBB distributions and percentile 95% confidence intervals; hybrid = mean across seeds and development time.','replicas_mbb/*_paired.csv')
pd.DataFrame(FIG_CATALOG).to_csv(OUT/'tabelas'/'catalogo_figuras_bilingue.csv',index=False)
